# Mounting

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#import geopandas as gpd
#from shapely.geometry import Polygon
#from shapely.geometry import Point
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.cm as cm
from sklearn import preprocessing
from sklearn.cluster import Birch
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from scipy.special import rel_entr, kl_div
from scipy.stats import entropy
import math
import time
import random
from sklearn.metrics import r2_score

import warnings
warnings.filterwarnings('ignore')

path_seoul_data = '/content/drive/MyDrive/Colab Notebooks/datasets/Seoul/'
path_lta_data = '/content/drive/MyDrive/Colab Notebooks/data_fusion_fcl/lta_data/'

path_data = '/content/drive/MyDrive/Colab Notebooks/Workspace/tour_generation/data_tour/'
path_figure = '/content/drive/MyDrive/Colab Notebooks/Workspace/tour_generation/figure_tour/'
path_result = '/content/drive/MyDrive/Colab Notebooks/Workspace/tour_generation/re1_result/'

def reduce_mem_usage(props):
    start_mem_usg = props.memory_usage().sum() / 1024**2
    #print("Memory usage of properties dataframe is :",start_mem_usg," MB")
    NAlist = [] # Keeps track of columns that have missing values filled in.
    for col in props.columns:
        if (props[col].dtype != object) & (props[col].dtype != 'category'):  # Exclude strings

            # make variables for Int, max and min
            IsInt = False
            mx = props[col].max()
            mn = props[col].min()

            # Integer does not support NA, therefore, NA needs to be filled
            if not np.isfinite(props[col]).all():
                NAlist.append(col)
                props[col].fillna(mn-1,inplace=True)

            # test if column can be converted to an integer
            asint = props[col].fillna(0).astype(np.int64)
            result = (props[col] - asint)
            result = result.sum()
            if result > -0.01 and result < 0.01:
                IsInt = True

            if IsInt:
                if mn >= 0:
                    if mx < 255:
                        props[col] = props[col].astype(np.uint8)
                    elif mx < 65535:
                        props[col] = props[col].astype(np.uint16)
                    elif mx < 4294967295:
                        props[col] = props[col].astype(np.uint32)
                    else:
                        props[col] = props[col].astype(np.uint64)
                else:
                    if mn > np.iinfo(np.int8).min and mx < np.iinfo(np.int8).max:
                        props[col] = props[col].astype(np.int8)
                    elif mn > np.iinfo(np.int16).min and mx < np.iinfo(np.int16).max:
                        props[col] = props[col].astype(np.int16)
                    elif mn > np.iinfo(np.int32).min and mx < np.iinfo(np.int32).max:
                        props[col] = props[col].astype(np.int32)
                    elif mn > np.iinfo(np.int64).min and mx < np.iinfo(np.int64).max:
                        props[col] = props[col].astype(np.int64)
            else:
                props[col] = props[col].astype(np.float32)

    mem_usg = props.memory_usage().sum() / 1024**2
    return props

Mounted at /content/drive


# Stage 1 Model Embeding

In [ ]:
# ============================================================
# Discrete-latent encoder-only fusion (non-param decoder)
#   + Embedding encoder (Φ = [D,S,E,C] integer indices)
#   + Sparse-per-Φ support for q(H|Φ)
# ============================================================

import os, math, json, re, random
from dataclasses import dataclass
from typing import Dict, Tuple, List, Optional, Callable

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

from sklearn.cluster import KMeans

# ------------------------------------------------------------
# 0) Utilities: feature builders & small helpers
# ------------------------------------------------------------

def reduce_mem_usage(df: pd.DataFrame) -> pd.DataFrame:
    """Downcast numerics to reduce RAM usage (simple & safe)."""
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtype
        if pd.api.types.is_numeric_dtype(col_type):
            c_min = df[col].min()
            c_max = df[col].max()
            if pd.api.types.is_float_dtype(col_type):
                df[col] = pd.to_numeric(df[col], downcast='float')
            else:
                df[col] = pd.to_numeric(df[col], downcast='integer')
    end_mem = df.memory_usage().sum() / 1024**2
    # print(f"[reduce_mem] {start_mem:.2f} -> {end_mem:.2f} MB")
    return df

def distance(df, att):
    """Great-circle distance (km). att = [lon_o, lon_d, lat_o, lat_d] (deg)."""
    lon1 = np.radians(df[att[0]]); lon2 = np.radians(df[att[1]])
    lat1 = np.radians(df[att[2]]); lat2 = np.radians(df[att[3]])
    dlon = lon2 - lon1; dlat = lat2 - lat1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 6371*2*np.arcsin(np.sqrt(a))


def land_use(df_hts_, df_pcm_):
    """DESTINATION_SUBZONE -> LU_0..LU_5. Fills missing PCM zones with HTS global averages."""
    # 1. Calculate probabilities from HTS
    df = df_hts_[['DESTINATION_SUBZONE','TRIP_PURPOSE']].copy()
    df['Numbers'] = 1.0
    df = df.groupby(['DESTINATION_SUBZONE','TRIP_PURPOSE'])['Numbers'].sum().reset_index()

    # Normalize per zone
    df['Zone_Total'] = df.groupby('DESTINATION_SUBZONE')['Numbers'].transform('sum')
    df['Prob'] = df['Numbers'] / df['Zone_Total']

    # Pivot to LU_0...LU_5
    df_lu = df.pivot(index='DESTINATION_SUBZONE', columns='TRIP_PURPOSE', values='Prob').reset_index()
    df_lu.columns.name = None
    df_lu = df_lu.rename(columns={i: f'LU_{i}' for i in range(6)})
    df_lu = df_lu.fillna(0.0)

    # 2. Handle missing zones using PCM universe
    pcm_zones = pd.DataFrame({'DESTINATION_SUBZONE': df_pcm_['DESTINATION_SUBZONE'].unique()})
    df_final = pd.merge(pcm_zones, df_lu, on='DESTINATION_SUBZONE', how='left')

    # 3. Fill NaNs with the average of existing zones
    lu_cols = [f'LU_{i}' for i in range(6)]
    avg_values = df_lu[lu_cols].mean()
    df_final[lu_cols] = df_final[lu_cols].fillna(avg_values)

    return df_final

def mode_share(df_hts_, df_pcm_):
    """DESTINATION_SUBZONE -> TRANSIT_RATIO. Fills missing PCM zones with global HTS mean."""
    # 1. Calculate transit ratio per zone from HTS
    df = df_hts_[['DESTINATION_SUBZONE','TRAVEL_MODE']].copy()
    df['is_transit'] = (df['TRAVEL_MODE'] == 1).astype(float)

    # Group by zone and get mean (which is the probability P(mode=1|zone))
    df_ms = df.groupby('DESTINATION_SUBZONE')['is_transit'].mean().reset_index()
    df_ms.rename(columns={'is_transit': 'TRANSIT_RATIO'}, inplace=True)

    # 2. Handle missing zones using PCM universe
    pcm_zones = pd.DataFrame({'DESTINATION_SUBZONE': df_pcm_['DESTINATION_SUBZONE'].unique()})
    df_final = pd.merge(pcm_zones, df_ms, on='DESTINATION_SUBZONE', how='left')

    # 3. Fill NaNs with the global average transit ratio from known zones
    global_avg = df_ms['TRANSIT_RATIO'].mean()
    df_final['TRANSIT_RATIO'] = df_final['TRANSIT_RATIO'].fillna(global_avg)

    return df_final


# ------------------------------------------------------------
# 0.1) Discrete-Φ helpers (bins + clusters → indices)
# ------------------------------------------------------------

@dataclass
class DiscreteSpec:
    n_dist: int = 6
    n_start: int = 8
    n_end: int = 8
    n_lu_mode_clusters: int = 16
    dist_binning: str = "quantile"  # or "uniform"
    time_binning: str = "quantile"  # or "uniform"

def _fit_bins(x: np.ndarray, n: int, kind: str) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    if kind == "uniform":
        lo, hi = np.nanmin(x), np.nanmax(x)
        edges = np.linspace(lo, hi, n + 1)
    else:
        qs = np.linspace(0, 100, n + 1)
        edges = np.percentile(x, qs)
        edges[0]  = min(edges[0],  x.min()) - 1e-9
        edges[-1] = max(edges[-1], x.max()) + 1e-9
    return edges

def _digitize(x: np.ndarray, edges: np.ndarray) -> np.ndarray:
    return np.clip(np.digitize(x, edges, right=False) - 1, 0, len(edges) - 2)

def fit_semantic_spec_on_hts(
    df_hts: pd.DataFrame,
    spec: DiscreteSpec,
    distance_col="TRIP_DISTANCE",
    start_col="TRIP_STARTTIME",
    end_col="TRIP_ENDTIME",
    lu_cols=("LU_0","LU_1","LU_2","LU_3","LU_4","LU_5"),
    tr_col="TRANSIT_RATIO",
    km_random_state=42
) -> Dict:
    dist_edges  = _fit_bins(df_hts[distance_col].values, spec.n_dist,  spec.dist_binning)
    start_edges = _fit_bins(df_hts[start_col].values,   spec.n_start, spec.time_binning)
    end_edges   = _fit_bins(df_hts[end_col].values,     spec.n_end,   spec.time_binning)

    feat_cols = list(lu_cols) + [tr_col]
    X = df_hts[feat_cols].to_numpy(dtype=float)

    eps = 1e-8
    mu = np.nanmean(X, axis=0)
    sd = np.nanstd(X, axis=0)
    sd = np.where(sd < eps, 1.0, sd)
    Xz = (X - mu) / sd

    kmeans = KMeans(n_clusters=spec.n_lu_mode_clusters, n_init="auto", random_state=km_random_state)
    kmeans.fit(Xz)

    return {
        "dist_edges": dist_edges.tolist(),
        "start_edges": start_edges.tolist(),
        "end_edges": end_edges.tolist(),
        "lu_cols": list(lu_cols),
        "tr_col": tr_col,
        "lu_norm": {"mean": mu.tolist(), "std": sd.tolist(), "eps": eps, "feat_order": feat_cols},
        "kmeans_centers": kmeans.cluster_centers_.tolist(),
        "phi_segments": {
            "n_dist": spec.n_dist,
            "n_start": spec.n_start,
            "n_end": spec.n_end,
            "n_cluster": spec.n_lu_mode_clusters
        }
    }

def apply_semantic_spec(
    df: pd.DataFrame,
    fitted: Dict,
    distance_col="TRIP_DISTANCE",
    start_col="TRIP_STARTTIME",
    end_col="TRIP_ENDTIME"
) -> pd.DataFrame:
    df = df.copy()
    dist_edges  = np.array(fitted["dist_edges"], dtype=float)
    start_edges = np.array(fitted["start_edges"], dtype=float)
    end_edges   = np.array(fitted["end_edges"], dtype=float)

    d_bin = _digitize(df[distance_col].to_numpy(dtype=float), dist_edges)
    s_bin = _digitize(df[start_col].to_numpy(dtype=float),    start_edges)
    e_bin = _digitize(df[end_col].to_numpy(dtype=float),      end_edges)

    lu_meta = fitted["lu_norm"]
    feat_cols = lu_meta["feat_order"]
    mu = np.array(lu_meta["mean"], dtype=float)
    sd = np.array(lu_meta["std"], dtype=float)
    eps = float(lu_meta.get("eps", 1e-8))

    X = df[feat_cols].to_numpy(dtype=float)
    sd_safe = np.where(sd < eps, 1.0, sd)
    Xz = (X - mu) / sd_safe

    centers = np.array(fitted["kmeans_centers"], dtype=float)
    d2 = ((Xz[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
    c_id = d2.argmin(axis=1)

    df["D_BIN"] = d_bin.astype(int)
    df["S_BIN"] = s_bin.astype(int)
    df["E_BIN"] = e_bin.astype(int)
    df["C_ID"]  = c_id.astype(int)
    return df

def build_phi_indices_from_bins(
    df: pd.DataFrame,
    cols=("D_BIN","S_BIN","E_BIN","C_ID")
) -> Tuple[pd.DataFrame, int]:
    """Store Φ as integer indices [D,S,E,C]."""
    df = df.copy()
    D = df[cols[0]].astype(int).to_numpy()
    S = df[cols[1]].astype(int).to_numpy()
    E = df[cols[2]].astype(int).to_numpy()
    C = df[cols[3]].astype(int).to_numpy()
    df["PHI_IDX"] = [[int(D[i]), int(S[i]), int(E[i]), int(C[i])] for i in range(len(df))]
    return df, 4  # four categorical fields

# ------------------------------------------------------------
# 1) Data containers (index-based Φ)
# ------------------------------------------------------------

class HTSDataset(Dataset):
    """One HTS row = (phi_idx[4], xy_id)."""
    def __init__(self, df_hts: pd.DataFrame, xy_ids: np.ndarray):
        self.phi_idx = np.vstack(df_hts["PHI_IDX"].to_numpy()).astype(np.int64)  # [N,4]
        self.xy = xy_ids.astype(np.int64)
    def __len__(self): return self.phi_idx.shape[0]
    def __getitem__(self, i):
        return self.phi_idx[i], self.xy[i]

class PCMDataset(Dataset):
    """One PCM row = (phi_idx[4], count)."""
    def __init__(self, df_pcm: pd.DataFrame):
        self.phi_idx = np.vstack(df_pcm["PHI_IDX"].to_numpy()).astype(np.int64)  # [N,4]
        self.count = df_pcm["COUNT"].to_numpy(dtype=np.float32)
    def __len__(self): return self.phi_idx.shape[0]
    def __getitem__(self, i):
        return self.phi_idx[i], self.count[i]

# ------------------------------------------------------------
# 2) Embedding Encoder (Φ indices -> embeddings -> MLP -> logits[K])
# ------------------------------------------------------------

class EncoderEmbed(nn.Module):
    """
    Embedding encoder: [D,S,E,C] indices -> concat(emb_D, emb_S, emb_E, emb_C) -> MLP -> logits[K].
    """
    def __init__(
        self,
        cardinals: Tuple[int, int, int, int],  # (n_dist, n_start, n_end, n_cluster)
        K: int,
        emb_dims: Tuple[int, int, int, int] = (16, 16, 16, 16),
        hidden: int = 256,
        num_layers: int = 2,
        dropout: float = 0.0
    ):
        super().__init__()
        nD, nS, nE, nC = map(int, cardinals)
        eD, eS, eE, eC = emb_dims

        self.emb_D = nn.Embedding(nD, eD)
        self.emb_S = nn.Embedding(nS, eS)
        self.emb_E = nn.Embedding(nE, eE)
        self.emb_C = nn.Embedding(nC, eC)

        in_dim = eD + eS + eE + eC
        layers: List[nn.Module] = []
        dims = [in_dim] + [hidden]*(num_layers-1) + [K]
        for i in range(len(dims)-2):
            layers += [nn.Linear(dims[i], dims[i+1]), nn.ReLU(inplace=True)]
            if dropout > 0:
                layers += [nn.Dropout(dropout)]
        layers += [nn.Linear(dims[-2], dims[-1])]
        self.net = nn.Sequential(*layers)

    def forward(self, phi_idx: torch.Tensor) -> torch.Tensor:
        """
        phi_idx: LongTensor [B,4] with columns [D,S,E,C].
        returns logits [B,K]
        """
        D = phi_idx[:, 0]; S = phi_idx[:, 1]; E = phi_idx[:, 2]; C = phi_idx[:, 3]
        z = torch.cat([self.emb_D(D), self.emb_S(S), self.emb_E(E), self.emb_C(C)], dim=-1)
        return self.net(z)

# ------------------------------------------------------------
# 3) Numerics, schedules, divergences, masked softmax
# ------------------------------------------------------------

def safe_log(x: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    return torch.log(x.clamp_min(eps))

def entropy_categorical(probs: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    return -(probs * safe_log(probs, eps)).sum(dim=-1)

def js_divergence(p: torch.Tensor, q: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    p = p / p.sum().clamp_min(eps)
    q = q / q.sum().clamp_min(eps)
    m = 0.5 * (p + q)
    kl_pm = (p * (safe_log(p, eps) - safe_log(m, eps))).sum()
    kl_qm = (q * (safe_log(q, eps) - safe_log(m, eps))).sum()
    return 0.5 * (kl_pm + kl_qm)

def softmax_tau(logits: torch.Tensor, tau: float) -> torch.Tensor:
    return F.softmax(logits / max(tau, 1e-6), dim=-1)

def get_tau(ep: int, total_ep: int, tau_start: float, tau_end: float, schedule: str = "linear") -> float:
    if total_ep <= 1: return tau_end
    t = (ep - 1) / (total_ep - 1)  # 0..1
    if schedule == "linear":
        s = t
    elif schedule == "cosine":
        s = 0.5 * (1 - math.cos(math.pi * t))
    elif schedule == "exp":
        s = 1 - math.exp(-5 * t)
    else:
        s = t
    return (1 - s) * tau_start + s * tau_end

# ------------------------------------------------------------
# 3.1) Sparse support policies & masked softmax
# ------------------------------------------------------------

SupportPolicy = Callable[[torch.Tensor, Optional[torch.Tensor]], torch.Tensor]
# signature: mask = policy(logits, phi_idx_b) -> bool tensor [B,K], True = allowed

def masked_softmax(logits: torch.Tensor,
                   mask: torch.Tensor,
                   tau: float = 1.0,
                   eps: float = 1e-12) -> torch.Tensor:
    B, K = logits.shape
    scaled = logits / max(tau, 1e-6)
    scaled = scaled.masked_fill(~mask, -1e9)
    q = F.softmax(scaled, dim=-1)
    q = q * mask.float()
    z = q.sum(dim=-1, keepdim=True).clamp_min(eps)
    return q / z

def topk_support_policy(k: int) -> SupportPolicy:
    def policy(logits: torch.Tensor, phi_idx_b: Optional[torch.Tensor] = None) -> torch.Tensor:
        B, K = logits.shape
        k_eff = min(max(1, k), K)
        idx = torch.topk(logits, k=k_eff, dim=-1).indices
        mask = torch.zeros_like(logits, dtype=torch.bool)
        mask.scatter_(1, idx, True)
        return mask
    return policy

def prob_threshold_policy(thr: float) -> SupportPolicy:
    def policy(logits: torch.Tensor, phi_idx_b: Optional[torch.Tensor] = None) -> torch.Tensor:
        p = F.softmax(logits, dim=-1)
        return p >= thr
    return policy

# ---- Bucketed support mining (index-based) ----

from collections import defaultdict

def build_phi_bucket_candidates_embed(
    encoder: nn.Module,
    hts_loader: DataLoader,
    K: int,
    tau: float,
    topk: int = 8,
    device: Optional[str] = None
) -> Dict[Tuple[int, int, int, int], np.ndarray]:
    """Aggregate q(H|Φ) over HTS and keep top-k H per Φ bucket. Φ is [D,S,E,C] indices."""
    enc_device = next(encoder.parameters()).device if device is None else torch.device(device)
    encoder.eval()
    acc = defaultdict(lambda: np.zeros(K, dtype=np.float64))

    with torch.no_grad():
        for phi_idx_b, _ in hts_loader:
            phi_idx_b = phi_idx_b.to(enc_device)              # [B,4] long
            logits = encoder(phi_idx_b)                       # [B,K]
            q = F.softmax(logits / max(tau, 1e-6), dim=-1).cpu().numpy()
            key_rows = phi_idx_b.cpu().numpy()                # [[D,S,E,C], ...]
            for i in range(key_rows.shape[0]):
                key = tuple(int(x) for x in key_rows[i])
                acc[key] += q[i]

    out = {}
    for key, vec in acc.items():
        k_eff = min(max(1, topk), K)
        keep = np.argpartition(-vec, k_eff-1)[:k_eff]
        keep = keep[np.argsort(-vec[keep])]
        out[key] = keep
    return out

def bucket_support_policy_embed(
    candidates: Dict[Tuple[int,int,int,int], np.ndarray]
) -> SupportPolicy:
    def policy(logits: torch.Tensor, phi_idx_b: Optional[torch.Tensor]) -> torch.Tensor:
        device = logits.device
        B, K = logits.shape
        mask = torch.zeros((B, K), dtype=torch.bool, device=device)
        keys = phi_idx_b.tolist()  # list of [D,S,E,C]
        for i in range(B):
            key = tuple(int(x) for x in keys[i])
            keep = candidates.get(key)
            if keep is None or len(keep) == 0:
                j = int(torch.argmax(logits[i]))
                mask[i, j] = True
            else:
                idx = torch.as_tensor(keep, dtype=torch.long, device=device)
                mask[i, idx] = True
        return mask
    return policy

# ------------------------------------------------------------
# 4) Non-param decoder  \hat{P}(XY|H)  (masked)
# ------------------------------------------------------------

@torch.no_grad()
def build_nonparam_decoder_xy_given_h(
    encoder: nn.Module,
    hts_loader: DataLoader,
    n_xy: int,
    K: int,
    device: str = "cpu",
    dtype: torch.dtype = torch.float32,
    progress: bool = True,
    tau: float = 1.0,
    support_policy: Optional[SupportPolicy] = None
) -> torch.Tensor:
    """
    \hat P(XY=j | H=h) = E[ 1{XY=j} q(h|Phi) ] / E[ q(h|Phi) ], with q masked per Φ (indices).
    Returns CPU tensor [n_xy, K] with columns summing to 1 (over kept H).
    """
    encoder.eval()
    num = torch.zeros((n_xy, K), dtype=dtype, device="cpu")
    den = torch.zeros((K,), dtype=dtype, device="cpu")

    for bi, (phi_idx_b, xy_b) in enumerate(hts_loader):
        if progress and bi % 50 == 0:
            print(f"[decoder] pass chunk {bi}")
        phi_idx_b = phi_idx_b.to(device=device, dtype=torch.long)
        logits = encoder(phi_idx_b)
        if support_policy is None:
            q = softmax_tau(logits, tau)
        else:
            mask = support_policy(logits, phi_idx_b)
            q = masked_softmax(logits, mask, tau)
        q = q.to("cpu")
        xy_b = xy_b.to("cpu")

        num.index_add_(0, xy_b, q)
        den += q.sum(dim=0)

    P = torch.zeros_like(num)
    mask = den > 0
    if mask.any():
        P[:, mask] = num[:, mask] / den[mask]
    P = P / P.sum(dim=0, keepdim=True).clamp_min(1e-12)
    return P

# ------------------------------------------------------------
# 5) Loss components (masked q everywhere)
# ------------------------------------------------------------

def compute_hts_nll(
    encoder: nn.Module,
    hts_loader: DataLoader,
    P_xy_given_h: torch.Tensor,
    log_norm: float,
    device: str,
    dtype: torch.dtype,
    progress: bool = True,
    tau: float = 1.0,
    support_policy: Optional[SupportPolicy] = None,
):
    total_nll = torch.zeros((), device=device, dtype=dtype)
    totalN = 0
    P_xy_given_h = P_xy_given_h.to(device=device, dtype=dtype)

    for bi, (phi_idx_b, xy_b) in enumerate(hts_loader):
        if progress and bi % 50 == 0:
            print(f"[hts-nll] pass chunk {bi}")
        B = phi_idx_b.shape[0]
        phi_idx_b = phi_idx_b.to(device=device, dtype=torch.long)
        xy_b  = xy_b.to(device=device, dtype=torch.long)

        logits = encoder(phi_idx_b)
        if support_policy is None:
            q = softmax_tau(logits, tau)
        else:
            mask = support_policy(logits, phi_idx_b)
            q = masked_softmax(logits, mask, tau)

        P_rows = P_xy_given_h.index_select(0, xy_b)    # [B, K]
        mix = (q * P_rows).sum(dim=-1).clamp_min(1e-12)
        nll_sum = -torch.log(mix).sum()

        total_nll = total_nll + nll_sum
        totalN += B

    avg_nll = total_nll / max(totalN, 1)
    if log_norm > 0:
        avg_nll = avg_nll / log_norm
    return avg_nll, totalN

def compute_latent_marginals(
    encoder: nn.Module,
    loader: DataLoader,
    K: int,
    device: str,
    dtype: torch.dtype,
    progress: bool = True,
    tau: float = 1.0,
    support_policy: Optional[SupportPolicy] = None,
):
    acc = torch.zeros(K, device=device, dtype=dtype)
    wsum = torch.zeros((), device=device, dtype=dtype)

    for bi, batch in enumerate(loader):
        if progress and bi % 50 == 0:
            print(f"[latent-marg] pass chunk {bi}")
        if len(batch) == 2:
            phi_idx_b, w_b = batch
            # HTS loader provides xy_b here; PCM loader provides COUNT; we handle both
            if w_b.dtype == torch.long or w_b.dtype == torch.int64:
                w_b = None
            else:
                w_b = w_b.to(device=device, dtype=dtype)
        else:
            phi_idx_b = batch[0]
            w_b = None

        phi_idx_b = phi_idx_b.to(device=device, dtype=torch.long)
        logits = encoder(phi_idx_b)
        if support_policy is None:
            q = softmax_tau(logits, tau)
        else:
            mask = support_policy(logits, phi_idx_b)
            q = masked_softmax(logits, mask, tau)

        if w_b is None:
            acc  = acc  + q.sum(dim=0)
            wsum = wsum + torch.tensor(q.shape[0], device=device, dtype=dtype)
        else:
            acc  = acc  + (q * w_b.unsqueeze(-1)).sum(dim=0)
            wsum = wsum + w_b.sum()

    p = acc / wsum.clamp_min(1e-12)
    p = p / p.sum().clamp_min(1e-12)
    return p

def compute_fusion_js(
    encoder: nn.Module,
    hts_loader: DataLoader,
    pcm_loader: DataLoader,
    K: int,
    device: str,
    dtype: torch.dtype,
    use_normalized: bool = True,
    tau: float = 1.0,
    support_policy: Optional[SupportPolicy] = None,
):
    p_hts = compute_latent_marginals(encoder, hts_loader, K, device, dtype, progress=False, tau=tau, support_policy=support_policy)
    p_pcm = compute_latent_marginals(encoder, pcm_loader, K, device, dtype, progress=False, tau=tau, support_policy=support_policy)
    m = 0.5 * (p_hts + p_pcm)
    js = 0.5 * (
        (p_hts * (torch.log(p_hts.clamp_min(1e-12)) - torch.log(m.clamp_min(1e-12)))).sum()
      + (p_pcm * (torch.log(p_pcm.clamp_min(1e-12)) - torch.log(m.clamp_min(1e-12)))).sum()
    )
    return js / math.log(2.0) if use_normalized else js

# ------------------------------------------------------------
# 6) Config
# ------------------------------------------------------------

@dataclass
class TrainConfig:
    K: int = 64
    enc_hidden: int = 256
    enc_layers: int = 2
    enc_dropout: float = 0.0
    epochs: int = 10
    batch_size_hts: int = 8192
    batch_size_pcm: int = 16384
    lr: float = 2e-3
    wd: float = 1e-4
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    use_normalized_objective: bool = True
    lambda_fus: float = 1.0
    dtype: torch.dtype = torch.float32
    rebuild_every: int = 1
    progress: bool = True

    # softmax temperature annealing
    tau_start: float = 1.0
    tau_end:   float = 0.3
    tau_schedule: str = "cosine"

    # support policy controls
    topk_h: int = 8                   # per-Φ top-k for q(H|Φ)
    use_bucket_candidates: bool = False
    bucket_topk: int = 8
    refresh_bucket_every: int = 2     # epochs; 0 disables refresh

# ------------------------------------------------------------
# 7) Training loops (masked)
# ------------------------------------------------------------

def _prepare_xy_and_loaders(df_hts: pd.DataFrame,
                            df_pcm: pd.DataFrame,
                            cfg: TrainConfig):
    att_xy_series = (df_hts["ATT_X"].astype(str) + "|" + df_hts["ATT_Y"].astype(str))
    xy_vals = att_xy_series.to_numpy()
    xy_unique = pd.unique(xy_vals)
    xy_map = {k: i for i, k in enumerate(xy_unique)}
    xy_ids = np.vectorize(xy_map.__getitem__)(xy_vals)
    xy_classes = list(xy_unique)
    n_xy = len(xy_classes)

    hts_ds = HTSDataset(df_hts, xy_ids)
    pcm_ds = PCMDataset(df_pcm)
    hts_loader_full = DataLoader(hts_ds, batch_size=cfg.batch_size_hts, shuffle=False, num_workers=0, pin_memory=False)
    pcm_loader_full = DataLoader(pcm_ds, batch_size=cfg.batch_size_pcm, shuffle=False, num_workers=0, pin_memory=False)
    phi_dim = 4  # indices length, not used by embedding encoder
    return n_xy, xy_map, xy_classes, hts_loader_full, pcm_loader_full, phi_dim

def _make_support_policy(enc: nn.Module,
                         hts_loader_full: DataLoader,
                         tau: float,
                         cfg: TrainConfig) -> SupportPolicy:
    if cfg.use_bucket_candidates:
        cands = build_phi_bucket_candidates_embed(
            enc, hts_loader_full, cfg.K, tau,
            topk=cfg.bucket_topk, device=cfg.device
        )
        return bucket_support_policy_embed(cands)
    else:
        return topk_support_policy(cfg.topk_h)

def train_encoder_only_fusion(
    df_hts: pd.DataFrame,
    df_pcm: pd.DataFrame,                 # kept for signature parity
    cfg: TrainConfig,
    phi_segments_meta: Dict[str,int]      # pass fitted["phi_segments"]
) -> Dict:
    device = cfg.device
    dtype = cfg.dtype

    n_xy, xy_map, xy_classes, hts_loader_full, _pcm_loader_full, phi_dim = _prepare_xy_and_loaders(df_hts, df_pcm, cfg)

    # Embedding encoder
    cardinals = (phi_segments_meta["n_dist"], phi_segments_meta["n_start"],
                 phi_segments_meta["n_end"],  phi_segments_meta["n_cluster"])
    enc = EncoderEmbed(
        cardinals=cardinals, K=cfg.K,
        emb_dims=(16,16,16,16),
        hidden=cfg.enc_hidden, num_layers=cfg.enc_layers, dropout=cfg.enc_dropout
    ).to(device=device, dtype=torch.float32)
    opt = torch.optim.AdamW(enc.parameters(), lr=cfg.lr, weight_decay=cfg.wd)

    logXY = math.log(max(n_xy, 2))

    # Initial tau and support policy
    tau = get_tau(1, cfg.epochs, cfg.tau_start, cfg.tau_end, cfg.tau_schedule)
    support_policy = _make_support_policy(enc, hts_loader_full, tau, cfg)

    # Initial non-param decoder
    P_xy_h = build_nonparam_decoder_xy_given_h(
        encoder=enc, hts_loader=hts_loader_full, n_xy=n_xy, K=cfg.K,
        device=device, dtype=dtype, progress=cfg.progress, tau=tau,
        support_policy=support_policy
    )  # CPU [n_xy, K]

    history = []
    for ep in range(1, cfg.epochs + 1):
        tau = get_tau(ep, cfg.epochs, cfg.tau_start, cfg.tau_end, cfg.tau_schedule)
        enc.train(); opt.zero_grad()

        # Optionally refresh bucket candidates (EM-style)
        if cfg.use_bucket_candidates and cfg.refresh_bucket_every > 0 and (ep == 1 or (ep % cfg.refresh_bucket_every) == 0):
            support_policy = _make_support_policy(enc, hts_loader_full, tau, cfg)

        # (A) HTS NLL only
        L_hts, _ = compute_hts_nll(
            encoder=enc, hts_loader=hts_loader_full,
            P_xy_given_h=P_xy_h,
            log_norm=(logXY if cfg.use_normalized_objective else 1.0),
            device=device, dtype=dtype, progress=cfg.progress, tau=tau,
            support_policy=support_policy
        )

        total = L_hts
        total.backward()
        opt.step()

        # (B) Rebuild decoder periodically with current tau + support
        if (ep % cfg.rebuild_every) == 0:
            P_xy_h = build_nonparam_decoder_xy_given_h(
                encoder=enc, hts_loader=hts_loader_full, n_xy=n_xy, K=cfg.K,
                device=device, dtype=dtype, progress=cfg.progress, tau=tau,
                support_policy=support_policy
            )

        metrics = {
            "epoch": ep,
            "tau": float(tau),
            "L_hts": float(L_hts.detach().cpu().item()),
            "J_total": float(total.detach().cpu().item()),
        }
        history.append(metrics)
        print(f"[epoch {ep:03d}] tau={tau:.4f}  L_hts={metrics['L_hts']:.6f}  total={metrics['J_total']:.6f}")

    # ===== Final evaluation with final tau + fresh support =====
    tau_final = get_tau(cfg.epochs, cfg.epochs, cfg.tau_start, cfg.tau_end, cfg.tau_schedule)
    support_policy = _make_support_policy(enc, hts_loader_full, tau_final, cfg)

    P_xy_h_final = build_nonparam_decoder_xy_given_h(
        encoder=enc, hts_loader=hts_loader_full, n_xy=n_xy, K=cfg.K,
        device=device, dtype=dtype, progress=cfg.progress, tau=tau_final,
        support_policy=support_policy
    )

    L_hts_final, _ = compute_hts_nll(
        encoder=enc, hts_loader=hts_loader_full, P_xy_given_h=P_xy_h_final,
        log_norm=(logXY if cfg.use_normalized_objective else 1.0),
        device=device, dtype=dtype, progress=False, tau=tau_final,
        support_policy=support_policy
    )

    J_final = float(L_hts_final.detach().cpu().item())
    print(f"[FINAL] K={cfg.K}  tau={tau_final:.4f}  L_hts={J_final:.6f}")

    pack = {
        "encoder_state_dict": enc.state_dict(),
        "encoder_cfg": {
            "cardinals": list(cardinals), "K": cfg.K, "hidden": cfg.enc_hidden,
            "layers": cfg.enc_layers, "dropout": cfg.enc_dropout, "dtype": str(dtype)
        },
        "xy_map": xy_map,
        "xy_classes": xy_classes,
        "P_xy_given_h": P_xy_h_final,         # CPU tensor [n_xy, K]
        "history": history,
        "config": cfg.__dict__,
        "tau_final": float(tau_final),
        "J_final": J_final,
        "L_hts_final": J_final,
        "phi_segments": dict(phi_segments_meta),
    }
    return {"model": enc, "pack": pack}

def train_encoder_only_fusion_finding_K(
    df_hts: pd.DataFrame,
    df_pcm: pd.DataFrame,
    cfg: TrainConfig,
    phi_segments_meta: Dict[str,int]
) -> Dict:
    device = cfg.device
    dtype = cfg.dtype

    # Prepare data/loaders
    n_xy, xy_map, xy_classes, hts_loader_full, pcm_loader_full, phi_dim = _prepare_xy_and_loaders(df_hts, df_pcm, cfg)

    # Build embedding encoder
    cardinals = (phi_segments_meta["n_dist"], phi_segments_meta["n_start"],
                 phi_segments_meta["n_end"],  phi_segments_meta["n_cluster"])
    enc = EncoderEmbed(
        cardinals=cardinals, K=cfg.K,
        emb_dims=(16,16,16,16),
        hidden=cfg.enc_hidden, num_layers=cfg.enc_layers, dropout=cfg.enc_dropout
    ).to(device=device, dtype=torch.float32)
    opt = torch.optim.AdamW(enc.parameters(), lr=cfg.lr, weight_decay=cfg.wd)

    logXY = math.log(max(n_xy, 2))

    # Init temperature + support policy
    tau = get_tau(1, cfg.epochs, cfg.tau_start, cfg.tau_end, cfg.tau_schedule)
    support_policy = _make_support_policy(enc, hts_loader_full, tau, cfg)

    # Initial non-parametric decoder P(XY|H)
    P_xy_h = build_nonparam_decoder_xy_given_h(
        encoder=enc, hts_loader=hts_loader_full, n_xy=n_xy, K=cfg.K,
        device=device, dtype=dtype, progress=cfg.progress, tau=tau,
        support_policy=support_policy
    )

    history = []
    for ep in range(1, cfg.epochs + 1):
        tau = get_tau(ep, cfg.epochs, cfg.tau_start, cfg.tau_end, cfg.tau_schedule)
        enc.train(); opt.zero_grad()

        # Optional refresh of Φ-bucket candidates
        if cfg.use_bucket_candidates and cfg.refresh_bucket_every > 0 and (ep == 1 or (ep % cfg.refresh_bucket_every) == 0):
            support_policy = _make_support_policy(enc, hts_loader_full, tau, cfg)

        # ---- Likelihood-only objective ----
        L_hts, _ = compute_hts_nll(
            encoder=enc, hts_loader=hts_loader_full,
            P_xy_given_h=P_xy_h,
            log_norm=(logXY if cfg.use_normalized_objective else 1.0),
            device=device, dtype=dtype, progress=cfg.progress, tau=tau,
            support_policy=support_policy
        )

        total = L_hts  # <-- only likelihood drives training
        total.backward()
        opt.step()

        # Periodically rebuild decoder with current encoder + tau + support
        if (ep % cfg.rebuild_every) == 0:
            P_xy_h = build_nonparam_decoder_xy_given_h(
                encoder=enc, hts_loader=hts_loader_full, n_xy=n_xy, K=cfg.K,
                device=device, dtype=dtype, progress=cfg.progress, tau=tau,
                support_policy=support_policy
            )

        metrics = {
            "epoch": ep,
            "tau": float(tau),
            "L_hts": float(L_hts.detach().cpu().item()),
            "J_total": float(total.detach().cpu().item()),  # equals L_hts
        }
        history.append(metrics)
        print(f"[epoch {ep:03d}] tau={tau:.4f}  L_hts={metrics['L_hts']:.6f}")

    # ===== Final evaluation =====
    tau_final = get_tau(cfg.epochs, cfg.epochs, cfg.tau_start, cfg.tau_end, cfg.tau_schedule)
    support_policy = _make_support_policy(enc, hts_loader_full, tau_final, cfg)

    # Final decoder
    P_xy_h_final = build_nonparam_decoder_xy_given_h(
        encoder=enc, hts_loader=hts_loader_full, n_xy=n_xy, K=cfg.K,
        device=device, dtype=dtype, progress=cfg.progress, tau=tau_final,
        support_policy=support_policy
    )

    # Final likelihood (objective)
    L_hts_final, _ = compute_hts_nll(
        encoder=enc, hts_loader=hts_loader_full, P_xy_given_h=P_xy_h_final,
        log_norm=(math.log(max(n_xy,2)) if cfg.use_normalized_objective else 1.0),
        device=device, dtype=dtype, progress=False, tau=tau_final,
        support_policy=support_policy
    )

    # Fusion consistency (post-hoc) = JS(p_HTS(H), p_PCM(H))
    L_fus_final = compute_fusion_js(
        encoder=enc, hts_loader=hts_loader_full, pcm_loader=pcm_loader_full,
        K=cfg.K, device=device, dtype=dtype,
        use_normalized=cfg.use_normalized_objective, tau=tau_final,
        support_policy=support_policy
    )

    # Print requested summaries
    print(f"[FINAL] K={cfg.K}  tau={tau_final:.4f}")
    print(f"        Final likelihood (HTS NLL): {float(L_hts_final):.6f}")
    print(f"        Fusion consistency (JS):    {float(L_fus_final):.6f}")

    pack = {
        "encoder_state_dict": enc.state_dict(),
        "encoder_cfg": {"cardinals": list(cardinals), "K": cfg.K, "hidden": cfg.enc_hidden,
                        "layers": cfg.enc_layers, "dropout": cfg.enc_dropout, "dtype": str(dtype)},
        "xy_map": xy_map,
        "xy_classes": xy_classes,
        "P_xy_given_h": P_xy_h_final,
        "history": history,
        "config": cfg.__dict__,
        "tau_final": float(tau_final),
        # store both metrics explicitly
        "L_hts_final": float(L_hts_final.detach().cpu().item()),
        "fusion_consistency_js": float(L_fus_final.detach().cpu().item()),
        "phi_segments": dict(phi_segments_meta),
    }
    return {"model": enc, "pack": pack}


# ------------------------------------------------------------
# 8) Decoder table -> DataFrame
# ------------------------------------------------------------

def pack_to_dataframe(pack: dict) -> pd.DataFrame:
    P_xy_h = pack["P_xy_given_h"]         # [n_xy, K] CPU torch.Tensor
    xy_classes = pack["xy_classes"]       # list of "ATT_X|ATT_Y"
    K = int(pack["encoder_cfg"]["K"])

    P = P_xy_h.detach().cpu().numpy()     # (n_xy, K)
    df = pd.DataFrame({"ATT_XY": xy_classes})
    h_cols = [f"H_{h}" for h in range(K)]
    df_probs = pd.DataFrame(P, columns=h_cols)
    df = pd.concat([df, df_probs], axis=1)
    df = df.melt(id_vars=["ATT_XY"], value_vars=h_cols,
                 var_name="H", value_name="PROB")
    df["H"] = df["H"].apply(lambda s: int(re.sub(r"^H_", "", s)))

    def split_att(x):
        parts = str(x).split("|", 1)
        if len(parts) == 2: return parts[0], parts[1]
        else: return parts[0], ""
    att = df["ATT_XY"].apply(split_att)
    df["ATT_X"] = att.apply(lambda t: t[0])
    df["ATT_Y"] = att.apply(lambda t: t[1])

    return df[["ATT_X", "ATT_Y", "H", "PROB"]].reset_index(drop=True)

# ------------------------------------------------------------
# 9) Latent diagnostics from q (supports sparsity via policies)
# ------------------------------------------------------------

def _latent_diagnostics_from_q(
    q_latent: np.ndarray,
    weights: Optional[np.ndarray],
    eps: float = 1e-12
) -> Dict[str, float]:
    N, K = q_latent.shape
    w = np.ones(N, dtype=np.float64) if weights is None else np.asarray(weights, dtype=np.float64)
    w = np.clip(w, 0.0, None)
    wsum = w.sum() + eps

    H = -(q_latent * np.log(np.clip(q_latent, eps, 1.0))).sum(axis=1)
    H_mean = float((H * w).sum() / wsum)
    H_norm = H_mean / math.log(K)

    p = (q_latent * w[:, None]).sum(axis=0) / wsum
    p = p / p.sum()
    H_marg = float(-(p * np.log(np.clip(p, eps, 1.0))).sum())
    I_est  = float(max(0.0, H_marg - H_mean))

    qmax = q_latent.max(axis=1)
    q2   = np.partition(q_latent, -2, axis=1)[:, -2]
    gap  = qmax - q2
    qmax_mean = float((qmax * w).sum() / wsum)
    gap_mean  = float((gap  * w).sum() / wsum)

    KL_qU_mean = float(math.log(K) - H_mean)
    KL_qU_norm = KL_qU_mean / math.log(K)

    winners = q_latent.argmax(axis=1)
    win_counts = np.bincount(winners, minlength=K).astype(np.float64)
    win_ratio_max = float(win_counts.max() / max(N, 1))

    return {
        "E[H(q)]": H_mean,
        "E[H(q)]/logK": H_norm,
        "H_marg": H_marg,
        "I_est = H(P) - E[H(q)]": I_est,
        "E[max q]": qmax_mean,
        "E[max-min2 gap]": gap_mean,
        "E[KL(q||U)]/logK": KL_qU_norm,
        "argmax_dominance_ratio": win_ratio_max,
        "K": float(K),
        "N": float(N),
    }

@torch.no_grad()
def _encode_q_from_df(
    df: 'pd.DataFrame',
    encoder: 'nn.Module',
    K: int,
    device: str,
    batch_size: int = 8192,
    tau: float = 1.0,
    weights_col: Optional[str] = None,
    support_policy: Optional[SupportPolicy] = None
) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    weights = None
    if (weights_col is not None) and (weights_col in df.columns):
        weights = df[weights_col].to_numpy(dtype=np.float64)

    phi_np = np.vstack(df["PHI_IDX"].to_numpy()).astype(np.int64)
    N = phi_np.shape[0]
    q_blocks = []

    encoder.eval()
    for start in range(0, N, batch_size):
        end = min(N, start + batch_size)
        phi_b = torch.from_numpy(phi_np[start:end]).to(device=device, dtype=torch.long)
        logits = encoder(phi_b)
        if support_policy is None:
            q = softmax_tau(logits, tau)
        else:
            mask = support_policy(logits, phi_b)
            q = masked_softmax(logits, mask, tau)
        q_blocks.append(q.cpu().numpy())

    q_latent = np.vstack(q_blocks)
    return q_latent, weights

def uniformity_report_from_df(
    df_hts: 'pd.DataFrame',
    df_pcm: 'pd.DataFrame',
    encoder: 'nn.Module',
    pack: Dict,
    device: str,
    batch_size: int = 8192,
    tau: Optional[float] = None,
    support_policy: Optional[SupportPolicy] = None,
) -> Dict[str, Dict[str, float]]:
    if tau is None:
        tau = float(pack.get("tau_final", 1.0))
    K = int(pack["encoder_cfg"]["K"])

    q_hts, _ = _encode_q_from_df(df_hts, encoder, K, device, batch_size, tau, weights_col=None, support_policy=support_policy)
    rep_hts = _latent_diagnostics_from_q(q_hts, weights=None)

    wcol = "COUNT" if "COUNT" in df_pcm.columns else None
    q_pcm, w_pcm = _encode_q_from_df(df_pcm, encoder, K, device, batch_size, tau, weights_col=wcol, support_policy=support_policy)
    rep_pcm = _latent_diagnostics_from_q(q_pcm, weights=w_pcm)

    return {"HTS": rep_hts, "PCM": rep_pcm}

# ------------------------------------------------------------
# 10) Merge-based latent encoding for PCM (masked export)
# ------------------------------------------------------------

@torch.no_grad()
def encode_pcm_to_latent_df22(
    df_pcm: pd.DataFrame,
    model: nn.Module,
    pack: Dict,
    device: str = "cpu",
    batch_size: int = 8192,
    tau: Optional[float] = None,
    topk_h: Optional[int] = None,
    h_prob_threshold: Optional[float] = None,
    support_policy: Optional[SupportPolicy] = None
) -> Tuple[pd.DataFrame, np.ndarray]:
    """
    Returns q_df columns: ["PCM_ROW","H","PROB"] where PROB = masked q_phi(H|Phi_row).
    If topk_h or threshold provided, additionally sparsifies the exported rows.
    """
    if tau is None:
        tau = float(pack.get("tau_final", 1.0))
    K = int(pack["encoder_cfg"]["K"])

    all_rows = []
    q_blocks = []

    N = len(df_pcm)
    model.eval()

    for start in range(0, N, batch_size):
        end = min(N, start + batch_size)
        phi_b = np.vstack(df_pcm["PHI_IDX"].iloc[start:end].to_numpy()).astype(np.int64)
        phi_b = torch.from_numpy(phi_b).to(device=device, dtype=torch.long)

        logits = model(phi_b)
        if support_policy is None:
            q = softmax_tau(logits, tau)
        else:
            mask = support_policy(logits, phi_b)
            q = masked_softmax(logits, mask, tau)

        q_cpu = q.to("cpu").numpy()
        q_blocks.append(q_cpu)

        B = q_cpu.shape[0]
        if (topk_h is None) and (h_prob_threshold is None):
            pcm_idx = np.repeat(np.arange(start, end), K)
            h_idx   = np.tile(np.arange(K), B)
            prob    = q_cpu.reshape(-1)
            block = np.column_stack([pcm_idx, h_idx, prob])
            all_rows.append(block)
        else:
            for i in range(B):
                probs = q_cpu[i]
                if h_prob_threshold is not None:
                    keep = np.where(probs >= h_prob_threshold)[0]
                    vals = probs[keep]
                else:
                    k = min(int(topk_h), K)
                    keep = np.argpartition(-probs, k-1)[:k]
                    keep = keep[np.argsort(-probs[keep])]
                    vals = probs[keep]
                pcm_row = start + i
                if len(keep) > 0:
                    block = np.column_stack([np.full_like(keep, pcm_row), keep, vals])
                    all_rows.append(block)

    q_latent = np.vstack(q_blocks) if len(q_blocks) > 0 else np.empty((0, K))
    if len(all_rows) == 0:
        q_df = pd.DataFrame(columns=["PCM_ROW", "H", "PROB"])
    else:
        arr = np.vstack(all_rows).astype(np.float64)
        q_df = pd.DataFrame(arr, columns=["PCM_ROW", "H", "PROB"]).astype(
            {"PCM_ROW": int, "H": int, "PROB": float}
        )
    return q_df, q_latent


def encode_pcm_to_latent_df(
    df_pcm: pd.DataFrame,
    model: nn.Module,
    pack: Dict,
    device: str = "cpu",
    batch_size: int = 8192,
    tau: Optional[float] = None,
    topk_h: int = 2, # Ensure we use top-k to keep it sparse
    support_policy: Optional[SupportPolicy] = None
) -> pd.DataFrame: # Removed q_latent return to save RAM

    if tau is None:
        tau = float(pack.get("tau_final", 1.0))
    K = int(pack["encoder_cfg"]["K"])

    all_pcm_rows = []
    all_h_indices = []
    all_probs = []

    N = len(df_pcm)
    model.eval()

    with torch.no_grad():
        for start in range(0, N, batch_size):
            end = min(N, start + batch_size)
            phi_b = np.vstack(df_pcm["PHI_IDX"].iloc[start:end].to_numpy()).astype(np.int64)
            phi_b = torch.from_numpy(phi_b).to(device=device, dtype=torch.long)

            logits = model(phi_b)
            if support_policy is None:
                q = softmax_tau(logits, tau)
            else:
                mask = support_policy(logits, phi_b)
                q = masked_softmax(logits, mask, tau)

            q_cpu = q.to("cpu").numpy().astype(np.float32) # Use float32

            # Efficiently extract top-k
            # Using argpartition is much faster than sorting the whole array
            for i in range(q_cpu.shape[0]):
                probs = q_cpu[i]
                k_eff = min(topk_h, K)

                # Get indices of top k probabilities
                keep = np.argpartition(-probs, k_eff-1)[:k_eff]
                # Sort them so the highest is first (optional but helpful)
                keep = keep[np.argsort(-probs[keep])]

                all_pcm_rows.extend([start + i] * k_eff)
                all_h_indices.extend(keep.tolist())
                all_probs.extend(probs[keep].tolist())

    # Build DataFrame directly from lists (more memory efficient than vstacking blocks)
    q_df = pd.DataFrame({
        "PCM_ROW": np.array(all_pcm_rows, dtype=np.int32),
        "H": np.array(all_h_indices, dtype=np.int32),
        "PROB": np.array(all_probs, dtype=np.float32)
    })

    return q_df

# ------------------------------------------------------------
# 11) Optional: conditional filtering helper
# ------------------------------------------------------------

def filter_by_conditioned_probability(
    df: pd.DataFrame,
    att_cols: List[str],
    prob_col: str = "Prob_XYZ_fus",
    *,
    abs_threshold: float = 1e-4,
    min_group_sum: float = 1e-15,
    add_column_name: str = "P_cond",
    return_splits: bool = True,
    preserve: str = "group_then_global"  # {"group_then_global", "global_only", "none"}
) -> Tuple[pd.DataFrame, Optional[pd.DataFrame]]:
    """
    Filter by per-group conditional probability with an absolute threshold.

    Steps:
      1) Compute group sums and conditional prob: P_cond = prob / group_sum
      2) Keep rows with P_cond >= abs_threshold
      3) Renormalize P_cond within each group to sum to 1
      4) Preserve probability mass:
         - "group_then_global": restore original group mass, then global normalize
         - "global_only": only global normalize
         - "none": leave raw group mass as-is (after step 3)
    Returns:
      kept_df, dropped_df (or None if return_splits=False)
    """
    if preserve not in {"group_then_global", "global_only", "none"}:
        raise ValueError("preserve must be one of {'group_then_global','global_only','none'}")

    df = df.copy()

    # 1) Group totals and conditional probability
    group_sum_orig = (
        df.groupby(att_cols, dropna=False)[prob_col]
          .transform("sum")
          .clip(lower=min_group_sum)
    )
    df[add_column_name] = df[prob_col] / group_sum_orig

    # 2) Keep rows above absolute conditional threshold
    keep_mask = df[add_column_name] >= float(abs_threshold)
    kept = df[keep_mask].reset_index(drop=True)
    dropped = df[~keep_mask].reset_index(drop=True) if return_splits else None

    if kept.empty:
        return kept, dropped

    # 3) Renormalize conditional probabilities within each group
    group_sum_kept_cond = (
        kept.groupby(att_cols, dropna=False)[add_column_name]
            .transform("sum")
            .clip(lower=min_group_sum)
    )
    kept[add_column_name] = kept[add_column_name] / group_sum_kept_cond

    # 4) Preserve probability mass as requested
    if preserve == "group_then_global":
        # Restore each group's original mass
        group_mass = (
            df[att_cols + [prob_col]]
            .groupby(att_cols, dropna=False)[prob_col]
            .sum()
            .rename("_GROUP_MASS_ORIG")
            .reset_index()
        )
        kept = kept.merge(group_mass, on=att_cols, how="left")
        kept["_GROUP_MASS_ORIG"] = kept["_GROUP_MASS_ORIG"].fillna(0.0)
        kept[prob_col] = kept[add_column_name] * kept["_GROUP_MASS_ORIG"]
        kept.drop(columns=["_GROUP_MASS_ORIG"], inplace=True)

        # Global normalization (optional but common in probability tables)
        total = float(kept[prob_col].sum())
        if total > 0:
            kept[prob_col] = kept[prob_col] / total

    elif preserve == "global_only":
        total = float(kept[prob_col].sum())
        if total > 0:
            kept[prob_col] = kept[prob_col] / total

    else:
        # "none": leave as-is; prob_col unchanged except for earlier steps
        pass

    return kept, dropped


# ------------------------------------------------------------
# 11) Validation: Marginal Distribution Comparison
# ------------------------------------------------------------

def validate_z_marginals(df_fused: pd.DataFrame, df_pcm_orig: pd.DataFrame, att_z_cols: list):
    print("\n" + "="*50)
    print("VALIDATION: Marginal Distribution of Z (PCM)")
    print("="*50)

    # 1. Prepare Ground Truth from original PCM
    # We use the 'COUNT' column to represent the true distribution
    gt_z = df_pcm_orig.groupby(att_z_cols)['COUNT'].sum().reset_index()
    gt_z['P_Z_true'] = gt_z['COUNT'] / gt_z['COUNT'].sum()

    # 2. Prepare Fused Marginal
    fus_z = df_fused.groupby(att_z_cols)['Prob_XYZ_fus'].sum().reset_index()
    fus_z.rename(columns={'Prob_XYZ_fus': 'P_Z_fused'}, inplace=True)

    # 3. Merge for comparison
    comparison = pd.merge(gt_z, fus_z, on=att_z_cols, how='outer').fillna(0)

    # 4. Calculate Metrics
    # Total Variation Distance: 0.5 * sum|p - q|
    tvd = 0.5 * np.abs(comparison['P_Z_true'] - comparison['P_Z_fused']).sum()

    # Pearson Correlation
    corr = np.corrcoef(comparison['P_Z_true'], comparison['P_Z_fused'])[0, 1]

    # Print Results
    print(f"Total Unique Z-bins (Trips): {len(comparison)}")
    print(f"Total Variation Distance (TVD): {tvd:.6f}  (Ideal: 0.0)")
    print(f"Pearson Correlation:           {corr:.6f}  (Ideal: 1.0)")

    # Summary of Drift
    max_drift = (comparison['P_Z_true'] - comparison['P_Z_fused']).abs().max()
    print(f"Max Probability Drift:         {max_drift:.6e}")

    if tvd < 0.01:
        print("RESULT: SUCCESS - P(att_Z) is highly preserved.")
    else:
        print("RESULT: WARNING - Significant drift detected in Z-marginal.")

# Stage 1 Data fusion

In [ ]:
# ------------------------------------------------------------
# __main__: Example end-to-end run (adjust paths)
# ------------------------------------------------------------
import os
import gc
import time
import random
import numpy as np
import pandas as pd
import torch
from typing import List, Optional, Tuple
from torch.utils.data import DataLoader, TensorDataset


def set_reproducible_seed(seed: int) -> None:
    """Set random seeds for Python, NumPy, PyTorch, CUDA, and cuDNN."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True

    torch.use_deterministic_algorithms(True, warn_only=True)


if __name__ == "__main__":

    # --------------------------------------------------------
    # Main parameters
    # --------------------------------------------------------
    nb = 10
    KK = 6000
    random_seed = 56789 #BASE_SEEDS = [12345, 23456, 34567 45678 56789]
    frac = 0.6

    # Set the seed before all preprocessing and model operations
    set_reproducible_seed(random_seed)

    start_time = time.time()
    reg = "sgp"

    severity = '10'
    scenario = 'combined'
    # --------------------------------------------------------
    # 1) Load data
    # --------------------------------------------------------
    df_hts = pd.read_csv(os.path.join(path_data, f"data_{reg}_hts_trip.csv"))
    #df_hts = pd.read_csv(os.path.join(path_data, f"data_{reg}_hts_trip_frac_{frac}.csv"))

    df_hts = df_hts[df_hts["INCOME"] < 3].copy()
    df_pcm = pd.read_csv(os.path.join(path_data, f"data_{reg}_pcm_trip.csv"))
    #df_pcm = pd.read_csv(os.path.join(path_data, f'data_{reg}_pcm_trip_severity_{severity}_{scenario}.csv'))

    age_mapping = {0: 0, 1: 0, 2: 1, 3: 1, 4: 2, 5: 2, 6: 3, 7: 3}
    df_hts["AGE"] = df_hts["AGE"].replace(age_mapping)

    if reg == 'seoul':
        df_pcm["AGE"] = df_pcm["AGE"].replace(age_mapping)

    # Aggregate PCM counts per (origin, destination, start, end)
    att_Z = [
        "ORIGIN_SUBZONE",
        "DESTINATION_SUBZONE",
        "TRIP_STARTTIME",
        "TRIP_ENDTIME"
    ]

    if "COUNT" not in df_pcm.columns:
        df_pcm["COUNT"] = 1.0
        df_pcm["COUNT"] = df_pcm.groupby(att_Z, observed=True)["COUNT"].transform("sum")
        df_pcm = df_pcm.drop_duplicates().reset_index(drop=True)

    # --------------------------------------------------------
    # 2) Construct base features for discretization
    # --------------------------------------------------------
    coordinate_cols = [
        "ORIGIN_SUBZONE_X",
        "DESTINATION_SUBZONE_X",
        "ORIGIN_SUBZONE_Y",
        "DESTINATION_SUBZONE_Y"
    ]

    df_hts["TRIP_DISTANCE"] = distance(df_hts, coordinate_cols)
    df_pcm["TRIP_DISTANCE"] = distance(df_pcm, coordinate_cols)

    df_land_use = land_use(df_hts, df_pcm)
    df_mode = mode_share(df_hts, df_pcm)

    df_hts = (
        df_hts
        .merge(df_land_use, on="DESTINATION_SUBZONE")
        .merge(df_mode, on="DESTINATION_SUBZONE")
    )
    df_pcm = (
        df_pcm
        .merge(df_land_use, on="DESTINATION_SUBZONE")
        .merge(df_mode, on="DESTINATION_SUBZONE")
    )

    # --------------------------------------------------------
    # 3) Fit and apply the discrete semantic specification
    # --------------------------------------------------------
    spec = DiscreteSpec(
        n_dist=nb,
        n_start=nb,
        n_end=nb,
        n_lu_mode_clusters=nb,
        dist_binning="uniform",
        time_binning="uniform"
    )

    # Resetting here makes any stochastic clustering in this step reproducible
    set_reproducible_seed(random_seed)

    fitted = fit_semantic_spec_on_hts(df_hts, spec)
    df_hts = apply_semantic_spec(df_hts, fitted)
    df_pcm = apply_semantic_spec(df_pcm, fitted)

    phi_segments_meta = fitted["phi_segments"]
    n_d = phi_segments_meta["n_dist"]
    n_s = phi_segments_meta["n_start"]
    n_e = phi_segments_meta["n_end"]
    n_c = phi_segments_meta["n_cluster"]

    df_hts, phi_dim = build_phi_indices_from_bins(df_hts)
    df_pcm, _ = build_phi_indices_from_bins(df_pcm)

    # --------------------------------------------------------
    # 4) Prepare attributes X, Y, and Z
    # --------------------------------------------------------
    att_X = ["AGE", "GENDER", "INCOME"]
    att_Y = ["TRIP_CNT", "TRIP_MAX", "TRIP_PURPOSE", "TRAVEL_MODE"]
    att_Z = [
        "ORIGIN_SUBZONE",
        "DESTINATION_SUBZONE",
        "TRIP_STARTTIME",
        "TRIP_ENDTIME"
    ]

    df_hts = df_hts[att_X + att_Y + att_Z + ["PHI_IDX"]].copy()
    df_pcm = df_pcm[att_Z + ["PHI_IDX", "COUNT"]].copy()

    df_hts["ATT_X"] = df_hts[att_X].astype(str).agg("_".join, axis=1)
    df_hts["ATT_Y"] = df_hts[att_Y].astype(str).agg("_".join, axis=1)
    df_hts["ATT_Z"] = df_hts[att_Z].astype(str).agg("_".join, axis=1)
    df_pcm["ATT_Z"] = df_pcm[att_Z].astype(str).agg("_".join, axis=1)

    df_hts = reduce_mem_usage(df_hts)
    df_pcm = reduce_mem_usage(df_pcm)

    # --------------------------------------------------------
    # 5) Train with sparse per-PHI support
    # --------------------------------------------------------
    cfg = TrainConfig(
        K=KK,
        enc_hidden=256,
        enc_layers=2,
        enc_dropout=0.0,
        epochs=5,
        batch_size_hts=8192,
        batch_size_pcm=16384,
        lr=2e-3,
        wd=1e-4,
        device=("cuda" if torch.cuda.is_available() else "cpu"),
        use_normalized_objective=True,
        rebuild_every=1,
        progress=True,
        tau_start=0.01,
        tau_end=0.01,
        tau_schedule="cosine",
        topk_h=10,
        use_bucket_candidates=False
        # bucket_topk=8,
        # refresh_bucket_every=2
    )

    # Reset immediately before data-loader creation and model initialization
    set_reproducible_seed(random_seed)

    out = train_encoder_only_fusion(
        df_hts=df_hts,
        df_pcm=df_pcm,
        cfg=cfg,
        phi_segments_meta=phi_segments_meta
    )
    model, pack = out["model"], out["pack"]

    # --------------------------------------------------------
    # 6) Build the decoder table once after training
    # --------------------------------------------------------
    decoder_df = pack_to_dataframe(pack).copy()
    decoder_df = decoder_df[decoder_df["PROB"] > 1e-4].copy()

    sum_h = decoder_df.groupby("H", observed=True)["PROB"].transform("sum")
    decoder_df = decoder_df[sum_h > 0].copy()
    decoder_df["PROB"] = decoder_df["PROB"] / sum_h[sum_h > 0]
    decoder_df.rename(columns={"PROB": "PROB_XY|H"}, inplace=True)

    # --------------------------------------------------------
    # 7) Prepare the same support policy for inference
    # --------------------------------------------------------
    tau_eval = float(pack.get("tau_final", 1.0))

    if cfg.use_bucket_candidates:
        phi_hts = torch.from_numpy(
            np.vstack(df_hts["PHI_IDX"].to_numpy()).astype(np.int64)
        )
        dummy_y = torch.zeros(phi_hts.shape[0], dtype=torch.long)

        hts_phi_loader = DataLoader(
            TensorDataset(phi_hts, dummy_y),
            batch_size=cfg.batch_size_hts,
            shuffle=False,
            num_workers=0
        )

        cand_map = build_phi_bucket_candidates_embed(
            model=model,
            hts_phi_loader=hts_phi_loader,
            K=cfg.K,
            tau=tau_eval,
            topk=cfg.bucket_topk,
            device=cfg.device
        )
        support_policy_infer = bucket_support_policy_embed(cand_map)
    else:
        support_policy_infer = topk_support_policy(cfg.topk_h)

    # --------------------------------------------------------
    # 8) Encode PCM observations into latent states
    # --------------------------------------------------------
    q_df = encode_pcm_to_latent_df(
        df_pcm=df_pcm,
        model=model,
        pack=pack,
        device=cfg.device,
        batch_size=8192,
        tau=tau_eval,
        topk_h=2,
        support_policy=support_policy_infer
    )
    q_df.rename(columns={"PROB": "PROB_H"}, inplace=True)

    # Renormalize within each PCM row:
    # sum_H q(H | PHI_row) = 1
    sum_row = q_df.groupby("PCM_ROW", observed=True)["PROB_H"].transform("sum")
    q_df = q_df[sum_row > 0].copy()
    q_df["PROB_H"] = q_df["PROB_H"] / sum_row[sum_row > 0]

    # --------------------------------------------------------
    # 9) Prepare PCM- and HTS-side fusion components
    # --------------------------------------------------------
    df_fus_pcm = df_pcm.copy()
    df_fus_pcm["P_Z"] = df_fus_pcm["COUNT"] / df_fus_pcm["COUNT"].sum()
    df_fus_pcm["PCM_ROW"] = df_fus_pcm.index
    df_fus_pcm = df_fus_pcm.merge(q_df, on="PCM_ROW")

    df_hts_attrs = df_hts[
        att_X + att_Y + ["ATT_X", "ATT_Y"]
    ].drop_duplicates()

    df_fus_hts = decoder_df.merge(
        df_hts_attrs,
        on=["ATT_X", "ATT_Y"]
    )

    # --------------------------------------------------------
    # 10) Identify cohorts
    # --------------------------------------------------------
    att_cols = ["AGE", "GENDER", "INCOME", "TRIP_MAX", "TRIP_CNT"]

    df_cohorts = (
        df_fus_hts[att_cols]
        .drop_duplicates()
        .sort_values(att_cols)
        .reset_index(drop=True)
    )

    fus_parts = []

    # --------------------------------------------------------
    # 11) Process each cohort
    # --------------------------------------------------------
    for age, gender, income, trip_max, trip_cnt in df_cohorts.itertuples(index=False):

        mask = (
            (df_fus_hts["AGE"] == age) &
            (df_fus_hts["GENDER"] == gender) &
            (df_fus_hts["INCOME"] == income) &
            (df_fus_hts["TRIP_MAX"] == trip_max) &
            (df_fus_hts["TRIP_CNT"] == trip_cnt)
        )

        df_hts_part = df_fus_hts.loc[mask].copy()

        if df_hts_part.empty:
            continue

        # Merge person-side and PCM-side distributions through latent state H
        df_part = df_hts_part.merge(df_fus_pcm, on="H")

        # P(X,Y,Z,H) = P(X,Y|H) P(H|Z) P(Z)
        df_part["Prob_XYZ_fus"] = (
            df_part["PROB_XY|H"] *
            df_part["PROB_H"] *
            df_part["P_Z"]
        )

        # Aggregate immediately to control RAM usage
        df_part = (
            df_part
            .groupby(att_X + att_Y + att_Z, observed=True)["Prob_XYZ_fus"]
            .sum()
            .reset_index()
        )

        # Remove negligible probabilities
        df_part = df_part[df_part["Prob_XYZ_fus"] > 1e-9].copy()

        if not df_part.empty:
            fus_parts.append(df_part)

        print(
            f"AGE={age}, GENDER={gender}, INCOME={income}, "
            f"TRIP_MAX={trip_max}, TRIP_CNT={trip_cnt}"
        )

        del df_hts_part, df_part
        gc.collect()

    if not fus_parts:
        raise RuntimeError(
            "Fusion produced no supported records. Check decoder threshold, "
            "topk_h, and PCM latent-state support."
        )

    # --------------------------------------------------------
    # 12) Merge cohorts
    # --------------------------------------------------------
    print("Merging cohorts and applying Z-preservation...")

    df_fusion = pd.concat(fus_parts, ignore_index=True)
    del fus_parts
    gc.collect()

    # Collapse duplicates that may appear in different cohorts
    df_fusion = (
        df_fusion
        .groupby(att_X + att_Y + att_Z, observed=True)["Prob_XYZ_fus"]
        .sum()
        .reset_index()
    )

    # --------------------------------------------------------
    # 13) Apply global Z-marginal preservation
    # --------------------------------------------------------
    df_pcm_targets = (
        df_pcm
        .groupby(att_Z, observed=True)["COUNT"]
        .sum()
        .reset_index()
    )
    df_pcm_targets["target_P_Z"] = (
        df_pcm_targets["COUNT"] / df_pcm_targets["COUNT"].sum()
    )

    df_fusion = df_fusion.merge(
        df_pcm_targets[att_Z + ["target_P_Z"]],
        on=att_Z,
        how="left",
        validate="many_to_one"
    )

    # Calculate current Z mass after the merge so row alignment is guaranteed
    z_mass_current = df_fusion.groupby(
        att_Z,
        observed=True
    )["Prob_XYZ_fus"].transform("sum")

    valid_z = (
        z_mass_current.gt(0) &
        df_fusion["target_P_Z"].notna()
    )
    df_fusion = df_fusion.loc[valid_z].copy()
    z_mass_current = z_mass_current.loc[valid_z]

    # Scale each Z group so that its fused mass equals the PCM target mass
    df_fusion["Prob_XYZ_fus"] *= (
        df_fusion["target_P_Z"] / z_mass_current
    )

    # Final cleanup and global normalization
    df_fusion.drop(columns=["target_P_Z"], inplace=True)

    total_probability = df_fusion["Prob_XYZ_fus"].sum()
    if not np.isfinite(total_probability) or total_probability <= 0:
        raise RuntimeError(
            f"Invalid final probability mass: {total_probability}."
        )

    df_fusion["Prob_XYZ_fus"] /= total_probability

    # --------------------------------------------------------
    # 14) Save once after all cohorts are completed
    # --------------------------------------------------------
    os.makedirs(path_result, exist_ok=True)

    output_file = os.path.join(path_result, f"case_{reg}_sim_trip_M_{KK}_nb_{nb}_seed_{random_seed}.csv")
    #output_file = os.path.join(path_result, f"case_{reg}_sim_trip_M_{KK}_nb_{nb}_seed_{random_seed}_frac_{frac}.csv")
    #output_file = os.path.join(path_result, f"case_{reg}_sim_trip_M_{KK}_nb_{nb}_seed_{random_seed}_severity_{severity}_{scenario}.csv")

    df_fusion.to_csv(output_file, index=False)

    # Optional validation
    # validate_z_marginals(df_fusion, df_pcm, att_Z)

    # --------------------------------------------------------
    # 15) Report completion
    # --------------------------------------------------------
    elapsed_time = time.time() - start_time

    print(f"Random seed: {random_seed}")
    print(f"Final probability sum: {df_fusion['Prob_XYZ_fus'].sum():.12f}")
    print(df_fusion.info())
    print(f"Output saved to: {output_file}")
    print(f"Elapsed time: {elapsed_time:.2f} seconds")

[decoder] pass chunk 0
[hts-nll] pass chunk 0
[decoder] pass chunk 0
[epoch 001] tau=0.0100  L_hts=0.593620  total=0.593620
[hts-nll] pass chunk 0
[decoder] pass chunk 0
[epoch 002] tau=0.0100  L_hts=0.553088  total=0.553088
[hts-nll] pass chunk 0
[decoder] pass chunk 0
[epoch 003] tau=0.0100  L_hts=0.536733  total=0.536733
[hts-nll] pass chunk 0
[decoder] pass chunk 0
[epoch 004] tau=0.0100  L_hts=0.529295  total=0.529295
[hts-nll] pass chunk 0
[decoder] pass chunk 0
[epoch 005] tau=0.0100  L_hts=0.522927  total=0.522927
[decoder] pass chunk 0
[FINAL] K=6000  tau=0.0100  L_hts=0.519848
AGE=0, GENDER=0, INCOME=0, TRIP_MAX=2, TRIP_CNT=1
AGE=0, GENDER=0, INCOME=0, TRIP_MAX=2, TRIP_CNT=2
AGE=0, GENDER=0, INCOME=0, TRIP_MAX=3, TRIP_CNT=1
AGE=0, GENDER=0, INCOME=0, TRIP_MAX=3, TRIP_CNT=2
AGE=0, GENDER=0, INCOME=0, TRIP_MAX=3, TRIP_CNT=3
AGE=0, GENDER=0, INCOME=2, TRIP_MAX=2, TRIP_CNT=1
AGE=0, GENDER=0, INCOME=2, TRIP_MAX=2, TRIP_CNT=2
AGE=0, GENDER=1, INCOME=0, TRIP_MAX=2, TRIP_CNT=1
AGE=0,

#Stage 1 Finding M

In [ ]:
import os, math, gc, torch, numpy as np, pandas as pd
from dataclasses import replace
from typing import Dict

BASE_SEEDS = [12345, 23456, 34567, 45678, 56789]


def set_reproducible_seed(seed: int) -> None:
    """Seed Python-independent NumPy/PyTorch sources used by this pipeline."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)

# ---------------------------
# Minimal, RAM-friendly train
# ---------------------------
def train_encoder_only_fusion_finding_K_MINRAM(
    df_hts: pd.DataFrame,
    df_pcm: pd.DataFrame,
    cfg: "TrainConfig",
    phi_segments_meta: Dict[str,int],
    seed: int,
    use_mixed_precision: bool = True,   # safe no-op on CPU; helpful on CUDA
) -> Dict[str, float]:
    """
    RAM-efficient variant:
      - No 'pack' and no model object returned.
      - No per-epoch history stored.
      - Temporary tensors are freed aggressively.
    Returns tiny dict of scalars: {"K", "L_hts_final", "fusion_consistency_js"}.
    """
    set_reproducible_seed(seed)
    device = cfg.device
    dtype = cfg.dtype

    # 1) Prepare data and loaders (do NOT keep extra returns around)
    n_xy, _, _, hts_loader_full, pcm_loader_full, _phi_dim = _prepare_xy_and_loaders(df_hts, df_pcm, cfg)

    # 2) Build encoder (embedding version)
    cardinals = (
        phi_segments_meta["n_dist"],
        phi_segments_meta["n_start"],
        phi_segments_meta["n_end"],
        phi_segments_meta["n_cluster"],
    )
    enc = EncoderEmbed(
        cardinals=cardinals,
        K=cfg.K,
        emb_dims=(16, 16, 16, 16),
        hidden=cfg.enc_hidden,
        num_layers=cfg.enc_layers,
        dropout=cfg.enc_dropout,
    ).to(device=device, dtype=torch.float32)  # keep params in fp32 for stability

    opt = torch.optim.AdamW(enc.parameters(), lr=cfg.lr, weight_decay=cfg.wd)
    logXY = math.log(max(n_xy, 2))

    # 3) Temperature & support policy
    tau = get_tau(1, cfg.epochs, cfg.tau_start, cfg.tau_end, cfg.tau_schedule)
    support_policy = _make_support_policy(enc, hts_loader_full, tau, cfg)

    # Initial non-parametric decoder; compute under no_grad and on CPU
    with torch.no_grad():
        P_xy_h = build_nonparam_decoder_xy_given_h(
            encoder=enc,
            hts_loader=hts_loader_full,
            n_xy=n_xy,
            K=cfg.K,
            device=device,
            dtype=dtype,
            progress=cfg.progress,
            tau=tau,
            support_policy=support_policy,
        )

    # 4) Train loop — no history stored
    scaler = torch.cuda.amp.GradScaler(enabled=(use_mixed_precision and torch.cuda.is_available()))
    last_rebuild_epoch = 0
    last_rebuild_tau = None
    for ep in range(1, cfg.epochs + 1):
        tau = get_tau(ep, cfg.epochs, cfg.tau_start, cfg.tau_end, cfg.tau_schedule)
        enc.train()
        opt.zero_grad(set_to_none=True)

        if (
            cfg.use_bucket_candidates
            and cfg.refresh_bucket_every > 0
            and (ep == 1 or (ep % cfg.refresh_bucket_every) == 0)
        ):
            support_policy = _make_support_policy(enc, hts_loader_full, tau, cfg)

        # Mixed precision autocast (safe no-op on CPU)
        autocast_enabled = (use_mixed_precision and torch.cuda.is_available())
        with torch.cuda.amp.autocast(enabled=autocast_enabled):
            L_hts, _ = compute_hts_nll(
                encoder=enc,
                hts_loader=hts_loader_full,
                P_xy_given_h=P_xy_h,
                log_norm=(logXY if cfg.use_normalized_objective else 1.0),
                device=device,
                dtype=dtype,
                progress=cfg.progress,
                tau=tau,
                support_policy=support_policy,
            )
            total = L_hts

        # Backprop + step
        if scaler.is_enabled():
            scaler.scale(total).backward()
            scaler.step(opt)
            scaler.update()
        else:
            total.backward()
            opt.step()

        # Rebuild decoder periodically without storing old copies
        if (ep % cfg.rebuild_every) == 0:
            with torch.no_grad():
                P_xy_h = build_nonparam_decoder_xy_given_h(
                    encoder=enc,
                    hts_loader=hts_loader_full,
                    n_xy=n_xy,
                    K=cfg.K,
                    device=device,
                    dtype=dtype,
                    progress=cfg.progress,
                    tau=tau,
                    support_policy=support_policy,
                )
            last_rebuild_epoch = ep
            last_rebuild_tau = tau

        print(f"[epoch {ep:03d}] tau={tau:.4f}  L_hts={float(L_hts.detach().cpu()):.6f}")

        # Free graph tensors. Do not call empty_cache every epoch: it discards
        # PyTorch's reusable CUDA memory pool and usually slows the next epoch.
        del L_hts, total

    # 5) Final evaluation — compute scalars only
    tau_final = get_tau(cfg.epochs, cfg.epochs, cfg.tau_start, cfg.tau_end, cfg.tau_schedule)
    support_policy = _make_support_policy(enc, hts_loader_full, tau_final, cfg)

    with torch.no_grad():
        # With a non-bucket top-k policy, if the decoder was rebuilt after the
        # final optimizer step at the same temperature, it is already the exact
        # final decoder. Reusing it removes one full HTS pass per K.
        can_reuse_decoder = (not cfg.use_bucket_candidates and last_rebuild_epoch == cfg.epochs
                             and last_rebuild_tau is not None
                             and abs(float(last_rebuild_tau) - float(tau_final)) < 1e-12)
        if can_reuse_decoder:
            P_xy_h_final = P_xy_h
        else:
            P_xy_h_final = build_nonparam_decoder_xy_given_h(
                encoder=enc, hts_loader=hts_loader_full, n_xy=n_xy, K=cfg.K,
                device=device, dtype=dtype, progress=False, tau=tau_final,
                support_policy=support_policy,
            )

        L_hts_final, _ = compute_hts_nll(
            encoder=enc,
            hts_loader=hts_loader_full,
            P_xy_given_h=P_xy_h_final,
            log_norm=(math.log(max(n_xy, 2)) if cfg.use_normalized_objective else 1.0),
            device=device,
            dtype=dtype,
            progress=False,
            tau=tau_final,
            support_policy=support_policy,
        )

        L_fus_final = compute_fusion_js(
            encoder=enc,
            hts_loader=hts_loader_full,
            pcm_loader=pcm_loader_full,
            K=cfg.K,
            device=device,
            dtype=dtype,
            use_normalized=cfg.use_normalized_objective,
            tau=tau_final,
            support_policy=support_policy,
        )

    print(f"[FINAL] K={cfg.K}  tau={tau_final:.4f}")
    print(f"        Final likelihood (HTS NLL): {float(L_hts_final):.6f}")
    print(f"        Fusion consistency (JS):    {float(L_fus_final):.6f}")

    # Pull out tiny scalars and immediately free everything heavy
    K_val  = int(cfg.K)
    L_val  = float(L_hts_final.detach().cpu().item())
    JS_val = float(L_fus_final.detach().cpu().item())

    # Drop references to big objects
    if P_xy_h_final is not P_xy_h:
        del P_xy_h_final
    del P_xy_h, enc, opt, support_policy
    del hts_loader_full, pcm_loader_full
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    return {"K": K_val, "seed": int(seed), "L_hts_final": L_val, "fusion_consistency_js": JS_val}


# ---------------------------
# Normalization & elbow (as-is)
# ---------------------------
def _minmax_norm(arr: np.ndarray) -> np.ndarray:
    arr = np.asarray(arr, dtype=np.float64)
    if arr.size == 0 or not np.isfinite(arr).all():
        raise ValueError("Metric array must be nonempty and contain only finite values.")
    amin, amax = float(arr.min()), float(arr.max())
    if amax - amin < 1e-12:
        return np.zeros_like(arr)
    return (arr - amin) / (amax - amin)

def find_elbow_by_max_distance(K_vals: np.ndarray, scores: np.ndarray) -> int:
    assert len(K_vals) == len(scores) and len(K_vals) >= 2
    x = _minmax_norm(np.asarray(K_vals, dtype=float))
    y = np.asarray(scores, dtype=float)
    x1, y1 = x[0], y[0]; x2, y2 = x[-1], y[-1]
    denom = np.hypot(y2 - y1, x2 - x1)
    if denom < 1e-12:
        return int(np.argmin(y))
    num = np.abs((y2 - y1) * x - (x2 - x1) * y + (x2 * y1 - y2 * x1))
    return int(np.argmax(num))


def mark_pareto_front(df: pd.DataFrame, col_l: str, col_js: str) -> np.ndarray:
    """Return True for nondominated rows when both criteria are minimized."""
    values = df[[col_l, col_js]].to_numpy(dtype=np.float64)
    pareto = np.ones(len(values), dtype=bool)
    for i, value in enumerate(values):
        dominated = np.all(values <= value, axis=1) & np.any(values < value, axis=1)
        pareto[i] = not dominated.any()
    return pareto


# ---------------------------
# K sweep — store metrics only
# ---------------------------
if __name__ == "__main__":
    reg = 'seoul'
    # --- Load data (unchanged) ---
    df_hts = pd.read_csv(os.path.join(path_data, f"data_{reg}_hts_trip.csv"))
    df_hts = df_hts[df_hts['INCOME'] < 3]

    df_pcm = pd.read_csv(os.path.join(path_data, f"data_{reg}_pcm_trip.csv"))

    # --- Age mapping ---
    age_map = dict(zip(range(8), [0, 0, 1, 1, 2, 2, 3, 3]))
    df_hts["AGE"] = df_hts["AGE"].replace(age_map)
    df_pcm["AGE"] = df_pcm["AGE"].replace(age_map)

    if "COUNT" not in df_pcm.columns:
        df_pcm["COUNT"] = 1.0
        df_pcm["COUNT"] = df_pcm.groupby(
            ["ORIGIN_SUBZONE","DESTINATION_SUBZONE","TRIP_STARTTIME","TRIP_ENDTIME"]
        )["COUNT"].transform("sum")
        df_pcm = df_pcm.drop_duplicates()

    # --- Build Φ and slim tables ---
    df_hts["TRIP_DISTANCE"] = distance(df_hts, ['ORIGIN_SUBZONE_X','DESTINATION_SUBZONE_X','ORIGIN_SUBZONE_Y','DESTINATION_SUBZONE_Y'])
    df_pcm["TRIP_DISTANCE"] = distance(df_pcm, ['ORIGIN_SUBZONE_X','DESTINATION_SUBZONE_X','ORIGIN_SUBZONE_Y','DESTINATION_SUBZONE_Y'])
    df_land_use = land_use(df_hts, df_pcm)
    df_mode = mode_share(df_hts, df_pcm)
    df_hts = df_hts.merge(df_land_use, on='DESTINATION_SUBZONE').merge(df_mode, on='DESTINATION_SUBZONE')
    df_pcm = df_pcm.merge(df_land_use, on='DESTINATION_SUBZONE').merge(df_mode, on='DESTINATION_SUBZONE')

    nb = 10
    spec = DiscreteSpec(n_dist=nb, n_start=nb, n_end=nb, n_lu_mode_clusters=nb,
                        dist_binning="uniform", time_binning="uniform")
    fitted = fit_semantic_spec_on_hts(df_hts, spec)
    df_hts = apply_semantic_spec(df_hts, fitted)
    df_pcm = apply_semantic_spec(df_pcm, fitted)
    phi_segments_meta = fitted["phi_segments"]

    df_hts, phi_dim = build_phi_indices_from_bins(df_hts)
    df_pcm, _       = build_phi_indices_from_bins(df_pcm)

    att_X = ['AGE', 'GENDER', 'INCOME']
    att_Y = ['TRIP_CNT','TRIP_MAX','TRIP_PURPOSE','TRAVEL_MODE']
    # K selection uses only X/Y labels, PHI_IDX and PCM COUNT. ATT_Z and the
    # coordinate columns are not consumed by the encoder-only objective, so
    # carrying/stringifying them wastes both time and RAM.
    df_hts = df_hts[att_X + att_Y + ['PHI_IDX']].copy()
    df_pcm = df_pcm[['PHI_IDX','COUNT']].copy()
    df_hts['ATT_X'] = df_hts[att_X].astype(str).agg('_'.join, axis=1)
    df_hts['ATT_Y'] = df_hts[att_Y].astype(str).agg('_'.join, axis=1)
    df_hts = reduce_mem_usage(df_hts)
    df_pcm = reduce_mem_usage(df_pcm)

    # --- Config baseline (unchanged) ---
    cfg_base = TrainConfig(
        K=3000, enc_hidden=256, enc_layers=2, enc_dropout=0.0,
        epochs=10, batch_size_hts=8192, batch_size_pcm=16384,
        lr=2e-3, wd=1e-4, device=("cuda" if torch.cuda.is_available() else "cpu"),
        use_normalized_objective=True, rebuild_every=2, progress=False,
        tau_start=0.01, tau_end=0.01, tau_schedule="cosine",
        topk_h=10, use_bucket_candidates=False,
    )

    # --- Sweep K over repeated deterministic seeds ---
    K_grid = [500, 1000, 2000, 3000, 4000, 6000, 8000, 10000]

    #K_grid = [i*500 for i in range(1,20)]
    os.makedirs(path_result, exist_ok=True)
    run_file = os.path.join(path_result, f"case_{reg}_K_sweep_runs_nb_{nb}.csv")
    summary_file = os.path.join(path_result, f"case_{reg}_K_sweep_summary_nb_{nb}.csv")
    rows = []
    for K in K_grid:
        cfgK = replace(cfg_base, K=K)
        for repeat, seed in enumerate(BASE_SEEDS, start=1):
            print(f"\n=== Running K={K}, repeat={repeat}/{len(BASE_SEEDS)}, seed={seed} ===")
            out = train_encoder_only_fusion_finding_K_MINRAM(
                df_hts=df_hts, df_pcm=df_pcm, cfg=cfgK, phi_segments_meta=phi_segments_meta,
                seed=seed, use_mixed_precision=True,
            )
            rows.append(out)
            # This checkpoint is tiny and prevents loss of completed runs if a
            # later K fails; it does not affect training or metric values.
            pd.DataFrame(rows).sort_values(["K", "seed"]).to_csv(run_file, index=False)
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    # Keep run-level results so stochastic uncertainty remains auditable.
    df_runs = pd.DataFrame(rows).sort_values(["K", "seed"]).reset_index(drop=True)
    n_rep = df_runs.groupby("K")["seed"].transform("count")
    if not (n_rep == len(BASE_SEEDS)).all():
        raise RuntimeError("At least one K does not contain all requested repetitions.")

    # Select K using mean metrics; SD and 95% CI quantify optimization variability.
    dfK = (df_runs.groupby("K", as_index=False)
           .agg(n_runs=("seed", "count"), L_hts_final_mean=("L_hts_final", "mean"),
                L_hts_final_sd=("L_hts_final", "std"),
                fusion_consistency_js_mean=("fusion_consistency_js", "mean"),
                fusion_consistency_js_sd=("fusion_consistency_js", "std"))
           .sort_values("K").reset_index(drop=True))
    root_n = np.sqrt(dfK["n_runs"].to_numpy(dtype=np.float64))
    dfK["L_hts_final_ci95"] = 1.96 * dfK["L_hts_final_sd"].fillna(0).to_numpy() / root_n
    dfK["fusion_js_ci95"] = 1.96 * dfK["fusion_consistency_js_sd"].fillna(0).to_numpy() / root_n
    dfK["L_hts_final_norm"] = _minmax_norm(dfK["L_hts_final_mean"].to_numpy())
    dfK["fusion_js_norm"] = _minmax_norm(dfK["fusion_consistency_js_mean"].to_numpy())
    dfK["score_norm"] = dfK["L_hts_final_norm"] + dfK["fusion_js_norm"]
    dfK["is_pareto"] = mark_pareto_front(dfK, "L_hts_final_mean", "fusion_consistency_js_mean")
    best_idx = int(dfK["score_norm"].to_numpy().argmin())
    best_K = int(dfK.loc[best_idx, "K"])
    dfK["is_selected"] = False
    dfK.loc[best_idx, "is_selected"] = True

    df_runs.to_csv(run_file, index=False)
    dfK.to_csv(summary_file, index=False)

    display_cols = ["K", "n_runs", "L_hts_final_mean", "L_hts_final_sd",
                    "fusion_consistency_js_mean", "fusion_consistency_js_sd",
                    "L_hts_final_norm", "fusion_js_norm", "score_norm", "is_pareto", "is_selected"]
    print("\n=== Repeated K-sweep summary ===")
    print(dfK[display_cols].to_string(index=False))
    print(f"\n>>> Selected K by minimum normalized mean loss = {best_K}")
    print(f">>> Run-level results: {run_file}")
    print(f">>> Summary results:   {summary_file}")

#Stage 2 Tour generation

In [ ]:
import os
import gc
import time
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ============================================================
# Configuration
# ============================================================

TRIP_MAX_ = 7

# Direct conditional sampling removes the need for very large
# intermediate oversampling. This controls batch generation only.
GENERATION_CHUNK_SIZE = 100_000

# If some sampled tours fail because a conditional transition is
# unsupported, additional batches are generated until target is met.
MAX_GENERATION_ROUNDS = 20

# BASE_SEEDS = [12345, 23456, 34567, 45678, 56789]
RANDOM_SEED = 12345

reg = "seoul"
KK = 6000
nb = 10
NN = 9_000_000

# ============================================================
# Memory helpers
# ============================================================

def _downcast_int(s: pd.Series):
    if s.isnull().any():
        return s
    x = s.astype(np.int64)
    xmin, xmax = x.min(), x.max()
    if xmin >= np.iinfo(np.int8).min and xmax <= np.iinfo(np.int8).max:
        return x.astype(np.int8)
    if xmin >= np.iinfo(np.int16).min and xmax <= np.iinfo(np.int16).max:
        return x.astype(np.int16)
    if xmin >= np.iinfo(np.int32).min and xmax <= np.iinfo(np.int32).max:
        return x.astype(np.int32)
    return x


def _downcast_float(s: pd.Series):
    return s.astype(np.float32)


def reduce_mem_usage(df: pd.DataFrame, use_cats: bool = False):
    """
    Downcast numeric columns.

    use_cats=False by default because repeated category conversions can
    themselves be expensive. Subzones are explicitly integer-encoded later.
    """
    for col in df.columns:
        if pd.api.types.is_integer_dtype(df[col]):
            df[col] = _downcast_int(df[col])
        elif pd.api.types.is_float_dtype(df[col]):
            df[col] = _downcast_float(df[col])
        elif use_cats and pd.api.types.is_object_dtype(df[col]):
            ratio = df[col].nunique(dropna=False) / max(len(df), 1)
            if ratio < 0.5:
                df[col] = df[col].astype("category")
    return df


# ============================================================
# General probability helpers
# ============================================================

def normalize_prob(prob):
    prob = np.asarray(prob, dtype=np.float64)
    s = prob.sum()
    if not np.isfinite(s) or s <= 0:
        return None
    return prob / s


def weighted_indices(prob, size, rng):
    prob = normalize_prob(prob)
    if prob is None or size <= 0:
        return np.empty(0, dtype=np.int64)
    return rng.choice(len(prob), size=int(size), replace=True, p=prob)


# ============================================================
# Coordinate / subzone encoding
# ============================================================

def build_subzone_encoding(df_fus, df_pcm):
    """
    Encode subzone labels as int32 during generation.

    This is substantially more memory-efficient than carrying strings or
    categorical labels through millions of synthetic schedules.
    """
    zones_fus = pd.concat(
        [df_fus["ORIGIN_SUBZONE"], df_fus["DESTINATION_SUBZONE"]],
        ignore_index=True
    )

    zones_pcm = pd.concat(
        [df_pcm["ORIGIN_SUBZONE"], df_pcm["DESTINATION_SUBZONE"]],
        ignore_index=True
    )

    zone_values = pd.concat([zones_fus, zones_pcm], ignore_index=True).dropna().unique()
    zone_categories = pd.Index(zone_values)

    print(f"Number of unique subzones: {len(zone_categories):,}")

    df_fus["ORIGIN_SUBZONE"] = pd.Categorical(
        df_fus["ORIGIN_SUBZONE"], categories=zone_categories
    ).codes.astype(np.int32)

    df_fus["DESTINATION_SUBZONE"] = pd.Categorical(
        df_fus["DESTINATION_SUBZONE"], categories=zone_categories
    ).codes.astype(np.int32)

    # --------------------------------------------------------
    # Build one coordinate record for each subzone
    # --------------------------------------------------------
    origin_coord = df_pcm[
        ["ORIGIN_SUBZONE", "ORIGIN_SUBZONE_X", "ORIGIN_SUBZONE_Y"]
    ].rename(columns={
        "ORIGIN_SUBZONE": "SUBZONE",
        "ORIGIN_SUBZONE_X": "X",
        "ORIGIN_SUBZONE_Y": "Y"
    })

    destination_coord = df_pcm[
        ["DESTINATION_SUBZONE", "DESTINATION_SUBZONE_X", "DESTINATION_SUBZONE_Y"]
    ].rename(columns={
        "DESTINATION_SUBZONE": "SUBZONE",
        "DESTINATION_SUBZONE_X": "X",
        "DESTINATION_SUBZONE_Y": "Y"
    })

    zone_coord = pd.concat(
        [origin_coord, destination_coord], ignore_index=True
    ).dropna(subset=["SUBZONE"]).drop_duplicates("SUBZONE", keep="first")

    coord_x = np.full(len(zone_categories), np.nan, dtype=np.float32)
    coord_y = np.full(len(zone_categories), np.nan, dtype=np.float32)

    coord_codes = pd.Categorical(
        zone_coord["SUBZONE"], categories=zone_categories
    ).codes

    valid = coord_codes >= 0

    coord_x[coord_codes[valid]] = zone_coord.loc[valid, "X"].to_numpy(
        dtype=np.float32
    )

    coord_y[coord_codes[valid]] = zone_coord.loc[valid, "Y"].to_numpy(
        dtype=np.float32
    )

    return df_fus, zone_categories, coord_x, coord_y


# ============================================================
# Next-start-time distribution
# ============================================================

def next_start_times(df_hts):
    """
    P(next trip start time | previous activity purpose,
                             previous trip end time)
    """
    t = df_hts[
        ["ACTIVITY_STARTTIME", "TRIP_PURPOSE", "ACTIVITY_DURATION"]
    ].copy()

    t["TRIP_STARTTIME"] = (
        t["ACTIVITY_STARTTIME"] + t["ACTIVITY_DURATION"]
    )

    t.rename(
        columns={"ACTIVITY_STARTTIME": "TRIP_ENDTIME"},
        inplace=True
    )

    t.drop(columns=["ACTIVITY_DURATION"], inplace=True)
    t["Numbers"] = 1

    grp_keys = [
        "TRIP_ENDTIME",
        "TRIP_PURPOSE",
        "TRIP_STARTTIME"
    ]

    agg = (
        t.groupby(grp_keys, observed=True, sort=False)["Numbers"]
        .sum()
        .reset_index()
    )

    base_keys = ["TRIP_ENDTIME", "TRIP_PURPOSE"]

    agg["Den"] = (
        agg.groupby(base_keys, observed=True, sort=False)["Numbers"]
        .transform("sum")
    )

    agg["Prob_cond"] = (
        agg["Numbers"] / agg["Den"]
    ).astype(np.float32)

    agg.drop(columns=["Numbers", "Den"], inplace=True)
    agg.dropna(inplace=True)

    return reduce_mem_usage(agg)


# ============================================================
# Build compact lookup for start-time sampling
# ============================================================

def build_next_start_lookup(df_next_start_times):
    """
    Dictionary:

        (previous purpose, previous end time)
            -> (possible next start times, probabilities)

    Number of dictionary entries is generally small because it is based
    on time-purpose combinations, not synthetic people.
    """
    lookup = {}

    grouped = df_next_start_times.groupby(
        ["TRIP_PURPOSE", "TRIP_ENDTIME"],
        observed=True,
        sort=False
    )

    for key, g in grouped:
        values = g["TRIP_STARTTIME"].to_numpy()
        probs = normalize_prob(g["Prob_cond"].to_numpy())

        if probs is not None:
            lookup[key] = (values, probs)

    return lookup


# ============================================================
# Cohort population targets
# ============================================================

def derive_cohort_targets(df_fus, N):
    """
    Derive AGE × GENDER × INCOME × TRIP_MAX target population directly
    from the probability mass of TRIP_CNT == 1.

    This replaces creation of 3*N = 18 million initial tours merely to
    estimate cohort proportions.
    """
    cohort_cols = ["AGE", "GENDER", "INCOME", "TRIP_MAX"]

    first = df_fus.loc[
        df_fus["TRIP_CNT"] == 1,
        cohort_cols + ["Prob_XYZ_fus"]
    ]

    dist = (
        first.groupby(cohort_cols, observed=True, sort=False)["Prob_XYZ_fus"]
        .sum()
        .reset_index(name="Mass")
    )

    total_mass = dist["Mass"].sum()

    if not np.isfinite(total_mass) or total_mass <= 0:
        raise RuntimeError("TRIP_CNT=1 has zero total probability mass.")

    dist["Pop"] = dist["Mass"] / total_mass
    raw = dist["Pop"].to_numpy(dtype=np.float64) * int(N)

    target = np.floor(raw).astype(np.int64)
    remaining = int(N) - int(target.sum())

    # Largest-remainder allocation guarantees exactly N people.
    if remaining > 0:
        remainder = raw - target
        add_idx = np.argsort(-remainder)[:remaining]
        target[add_idx] += 1

    dist["Target"] = target
    dist.drop(columns=["Mass"], inplace=True)

    return dist


# ============================================================
# First trip sampling
# ============================================================

def sample_first_trip(df_fus_cohort, n, rng):
    first = df_fus_cohort.loc[
        df_fus_cohort["TRIP_CNT"] == 1,
        [
            "AGE", "GENDER", "INCOME", "TRIP_MAX",
            "TRIP_PURPOSE", "TRAVEL_MODE",
            "ORIGIN_SUBZONE", "DESTINATION_SUBZONE",
            "TRIP_STARTTIME", "TRIP_ENDTIME",
            "Prob_XYZ_fus"
        ]
    ]

    if first.empty or n <= 0:
        return pd.DataFrame()

    idx = weighted_indices(
        first["Prob_XYZ_fus"].to_numpy(),
        n,
        rng
    )

    if len(idx) == 0:
        return pd.DataFrame()

    base_cols = [
        "AGE", "GENDER", "INCOME", "TRIP_MAX",
        "TRIP_PURPOSE", "TRAVEL_MODE",
        "ORIGIN_SUBZONE", "DESTINATION_SUBZONE",
        "TRIP_STARTTIME", "TRIP_ENDTIME"
    ]

    out = first.iloc[idx][base_cols].reset_index(drop=True)

    out.rename(columns={
        "TRIP_PURPOSE": "TRIP_PURPOSE_1",
        "TRAVEL_MODE": "TRAVEL_MODE_1",
        "ORIGIN_SUBZONE": "ORIGIN_SUBZONE_1",
        "DESTINATION_SUBZONE": "DESTINATION_SUBZONE_1",
        "TRIP_STARTTIME": "TRIP_STARTTIME_1",
        "TRIP_ENDTIME": "TRIP_ENDTIME_1"
    }, inplace=True)

    return out


# ============================================================
# Prepare next-trip distribution for one cohort
# ============================================================

def prepare_trip_lookup(df_fus_cohort, trip_idx):
    """
    For a fixed AGE/GENDER/INCOME/TRIP_MAX cohort:

        P(next trip | origin, start time)

    Demographic variables do not need to remain in the dictionary key
    because df_fus_cohort already fixes them.
    """
    cols = [
        "TRIP_PURPOSE",
        "TRAVEL_MODE",
        "ORIGIN_SUBZONE",
        "DESTINATION_SUBZONE",
        "TRIP_STARTTIME",
        "TRIP_ENDTIME",
        "Prob_XYZ_fus"
    ]

    table = df_fus_cohort.loc[
        df_fus_cohort["TRIP_CNT"] == trip_idx,
        cols
    ].copy()

    if table.empty:
        return None, None

    keys = ["ORIGIN_SUBZONE", "TRIP_STARTTIME"]

    table["Den"] = (
        table.groupby(keys, observed=True, sort=False)["Prob_XYZ_fus"]
        .transform("sum")
    )

    valid = table["Den"] > 0
    table = table.loc[valid].reset_index(drop=True)

    if table.empty:
        return None, None

    table["Prob"] = (
        table["Prob_XYZ_fus"] / table["Den"]
    ).astype(np.float32)

    table.drop(columns=["Prob_XYZ_fus", "Den"], inplace=True)

    lookup = {}

    groups = table.groupby(
        keys,
        observed=True,
        sort=False
    ).indices

    prob_values = table["Prob"].to_numpy(dtype=np.float64)

    for key, positions in groups.items():
        positions = np.asarray(positions, dtype=np.int64)

        p = normalize_prob(prob_values[positions])

        if p is not None:
            lookup[key] = (positions, p)

    return table, lookup


# ============================================================
# Sample next start time
# ============================================================

def sample_next_start(active, next_start_lookup, trip_idx, rng):
    """
    Sample one next-start time for each active person without materializing
    every possible start-time alternative.
    """
    n = len(active)

    if n == 0:
        return active

    purpose_col = f"TRIP_PURPOSE_{trip_idx - 1}"
    end_col = f"TRIP_ENDTIME_{trip_idx - 1}"

    # Use float64 because time columns may be integer or floating-point.
    sampled_start = np.full(n, np.nan, dtype=np.float64)
    valid = np.zeros(n, dtype=bool)

    groups = active.groupby(
        [purpose_col, end_col],
        observed=True,
        sort=False
    ).indices

    for key, positions in groups.items():
        item = next_start_lookup.get(key)

        if item is None:
            continue

        values, probs = item
        positions = np.asarray(positions, dtype=np.int64)

        sampled_start[positions] = rng.choice(
            values,
            size=len(positions),
            replace=True,
            p=probs
        )

        valid[positions] = True

    if not valid.all():
        active = active.loc[valid].reset_index(drop=True)
        sampled_start = sampled_start[valid]

    if active.empty:
        return active

    active["_NEXT_START"] = sampled_start

    return active


# ============================================================
# Sample next trip
# ============================================================

def sample_next_trip(active, trip_table, trip_lookup, trip_idx, rng):
    """
    Sample one next trip for each active schedule.

    This replaces the RAM-heavy many-to-many merge:

        partial tours × possible start times × possible next trips

    with grouped conditional sampling.
    """
    if active.empty or trip_table is None or not trip_lookup:
        return active.iloc[0:0].copy()

    previous_destination = f"DESTINATION_SUBZONE_{trip_idx - 1}"

    groups = active.groupby(
        [previous_destination, "_NEXT_START"],
        observed=True,
        sort=False
    ).indices

    n = len(active)
    valid = np.zeros(n, dtype=bool)
    selected = np.full(n, -1, dtype=np.int64)

    for key, positions in groups.items():
        item = trip_lookup.get(key)

        if item is None:
            continue

        candidate_positions, probs = item
        positions = np.asarray(positions, dtype=np.int64)

        selected[positions] = rng.choice(
            candidate_positions,
            size=len(positions),
            replace=True,
            p=probs
        )

        valid[positions] = True

    if not valid.any():
        return active.iloc[0:0].copy()

    if not valid.all():
        active = active.loc[valid].reset_index(drop=True)
        selected = selected[valid]

    source_columns = [
        "TRIP_PURPOSE",
        "TRAVEL_MODE",
        "ORIGIN_SUBZONE",
        "DESTINATION_SUBZONE",
        "TRIP_STARTTIME",
        "TRIP_ENDTIME"
    ]

    sampled = trip_table.iloc[selected][source_columns].reset_index(drop=True)

    active[f"TRIP_PURPOSE_{trip_idx}"] = sampled["TRIP_PURPOSE"].to_numpy()
    active[f"TRAVEL_MODE_{trip_idx}"] = sampled["TRAVEL_MODE"].to_numpy()
    active[f"ORIGIN_SUBZONE_{trip_idx}"] = sampled["ORIGIN_SUBZONE"].to_numpy()
    active[f"DESTINATION_SUBZONE_{trip_idx}"] = sampled["DESTINATION_SUBZONE"].to_numpy()
    active[f"TRIP_STARTTIME_{trip_idx}"] = sampled["TRIP_STARTTIME"].to_numpy()
    active[f"TRIP_ENDTIME_{trip_idx}"] = sampled["TRIP_ENDTIME"].to_numpy()

    active.drop(columns=["_NEXT_START"], inplace=True, errors="ignore")

    return active


# ============================================================
# Generate one batch for one demographic cohort
# ============================================================

def generate_cohort_batch(
    df_fus_cohort,
    tmax,
    batch_size,
    next_start_lookup,
    trip_tables,
    trip_lookups,
    rng
):
    active = sample_first_trip(
        df_fus_cohort,
        batch_size,
        rng
    )

    if active.empty:
        return active

    if tmax == 1:
        return active

    for trip_idx in range(2, int(tmax) + 1):
        active = sample_next_start(
            active,
            next_start_lookup,
            trip_idx,
            rng
        )

        if active.empty:
            break

        active = sample_next_trip(
            active,
            trip_tables.get(trip_idx),
            trip_lookups.get(trip_idx),
            trip_idx,
            rng
        )

        if active.empty:
            break

    if active.empty:
        return active

    # --------------------------------------------------------
    # Same home-return rule as original code:
    # if final trip purpose == 0, force final destination to
    # the first origin.
    # --------------------------------------------------------
    purpose_col = f"TRIP_PURPOSE_{tmax}"
    dest_col = f"DESTINATION_SUBZONE_{tmax}"

    if purpose_col in active.columns:
        home_mask = active[purpose_col] == 0

        if home_mask.any():
            active.loc[
                home_mask,
                dest_col
            ] = active.loc[
                home_mask,
                "ORIGIN_SUBZONE_1"
            ].to_numpy()

    return active.reset_index(drop=True)


# ============================================================
# Output-column helpers
# ============================================================

def schedule_columns_without_coordinates():
    cols = ["ID", "AGE", "GENDER", "INCOME", "TRIP_MAX"]

    for i in range(1, TRIP_MAX_ + 1):
        cols.extend([
            f"TRIP_PURPOSE_{i}",
            f"TRAVEL_MODE_{i}",
            f"ORIGIN_SUBZONE_{i}",
            f"DESTINATION_SUBZONE_{i}",
            f"TRIP_STARTTIME_{i}",
            f"TRIP_ENDTIME_{i}"
        ])

    return cols


def schedule_columns_with_coordinates():
    cols = ["ID", "AGE", "GENDER", "INCOME", "TRIP_MAX"]

    for i in range(1, TRIP_MAX_ + 1):
        cols.extend([
            f"TRIP_PURPOSE_{i}",
            f"TRAVEL_MODE_{i}",
            f"ORIGIN_SUBZONE_{i}",
            f"ORIGIN_SUBZONE_X_{i}",
            f"ORIGIN_SUBZONE_Y_{i}",
            f"DESTINATION_SUBZONE_{i}",
            f"DESTINATION_SUBZONE_X_{i}",
            f"DESTINATION_SUBZONE_Y_{i}",
            f"TRIP_STARTTIME_{i}",
            f"TRIP_ENDTIME_{i}"
        ])

    return cols


# ============================================================
# Restore labels and coordinates only for final output
# ============================================================

def restore_coordinates_and_labels(
    df,
    zone_categories,
    coord_x,
    coord_y
):
    """
    Add coordinates only after tour generation.

    During generation each subzone occupies only int32 rather than
    strings + four coordinate columns per trip.
    """
    for i in range(1, TRIP_MAX_ + 1):
        ocol = f"ORIGIN_SUBZONE_{i}"
        dcol = f"DESTINATION_SUBZONE_{i}"

        ox = f"ORIGIN_SUBZONE_X_{i}"
        oy = f"ORIGIN_SUBZONE_Y_{i}"
        dx = f"DESTINATION_SUBZONE_X_{i}"
        dy = f"DESTINATION_SUBZONE_Y_{i}"

        # ----------------------------------------------------
        # Missing trips
        # ----------------------------------------------------
        if ocol not in df.columns:
            df[ocol] = np.nan
            df[ox] = np.nan
            df[oy] = np.nan
        else:
            codes = pd.to_numeric(
                df[ocol],
                errors="coerce"
            ).to_numpy(dtype=np.float64)

            valid = (
                np.isfinite(codes) &
                (codes >= 0) &
                (codes < len(zone_categories))
            )

            x = np.full(len(df), np.nan, dtype=np.float32)
            y = np.full(len(df), np.nan, dtype=np.float32)

            icodes = np.zeros(len(df), dtype=np.int32)
            icodes[valid] = codes[valid].astype(np.int32)

            x[valid] = coord_x[icodes[valid]]
            y[valid] = coord_y[icodes[valid]]

            df[ox] = x
            df[oy] = y

            cat_codes = np.full(len(df), -1, dtype=np.int32)
            cat_codes[valid] = icodes[valid]

            df[ocol] = pd.Categorical.from_codes(
                cat_codes,
                categories=zone_categories
            )

        if dcol not in df.columns:
            df[dcol] = np.nan
            df[dx] = np.nan
            df[dy] = np.nan
        else:
            codes = pd.to_numeric(
                df[dcol],
                errors="coerce"
            ).to_numpy(dtype=np.float64)

            valid = (
                np.isfinite(codes) &
                (codes >= 0) &
                (codes < len(zone_categories))
            )

            x = np.full(len(df), np.nan, dtype=np.float32)
            y = np.full(len(df), np.nan, dtype=np.float32)

            icodes = np.zeros(len(df), dtype=np.int32)
            icodes[valid] = codes[valid].astype(np.int32)

            x[valid] = coord_x[icodes[valid]]
            y[valid] = coord_y[icodes[valid]]

            df[dx] = x
            df[dy] = y

            cat_codes = np.full(len(df), -1, dtype=np.int32)
            cat_codes[valid] = icodes[valid]

            df[dcol] = pd.Categorical.from_codes(
                cat_codes,
                categories=zone_categories
            )

        # Other missing trip attributes
        for col in [
            f"TRIP_PURPOSE_{i}",
            f"TRAVEL_MODE_{i}",
            f"TRIP_STARTTIME_{i}",
            f"TRIP_ENDTIME_{i}"
        ]:
            if col not in df.columns:
                df[col] = np.nan

    return df[schedule_columns_with_coordinates()]


# ============================================================
# Generate the complete final population in memory
# ============================================================

def generate_population(
    df_fus,
    cohort_targets,
    next_start_lookup,
    zone_categories,
    coord_x,
    coord_y,
    N,
    rng
):
    cohort_cols = ["AGE", "GENDER", "INCOME", "TRIP_MAX"]

    target_map = {
        tuple(row[c] for c in cohort_cols): int(row["Target"])
        for _, row in cohort_targets.iterrows()
    }

    population_parts = []
    total_generated = 0
    global_id = 0
    generation_seconds = 0.0

    print("\n" + "=" * 100)
    print("GENERATING POPULATION")
    print("=" * 100)

    # --------------------------------------------------------
    # Critical optimization:
    # group df_fus ONCE rather than repeatedly scanning the
    # full fused table for every cohort.
    # --------------------------------------------------------
    grouped_fus = df_fus.groupby(
        cohort_cols,
        observed=True,
        sort=False
    )

    n_cohorts = len(target_map)
    cohort_number = 0

    for cohort_key, df_fus_cohort in grouped_fus:
        if not isinstance(cohort_key, tuple):
            cohort_key = (cohort_key,)

        target = target_map.get(tuple(cohort_key), 0)

        if target <= 0:
            continue

        cohort_number += 1

        age, gender, income, tmax = cohort_key
        tmax = int(tmax)

        print(
            f"\n[COHORT {cohort_number}/{n_cohorts}] "
            f"AGE={age}, GENDER={gender}, INCOME={income}, "
            f"TRIP_MAX={tmax}, TARGET={target:,}"
        )

        if tmax < 1 or tmax > TRIP_MAX_:
            print(
                f"  WARNING: TRIP_MAX={tmax} outside supported "
                f"range 1..{TRIP_MAX_}. Cohort skipped."
            )
            continue

        # ----------------------------------------------------
        # Prepare each next-trip distribution exactly once
        # for the current cohort.
        # ----------------------------------------------------
        trip_tables = {}
        trip_lookups = {}

        supported = True

        for trip_idx in range(2, tmax + 1):
            table, lookup = prepare_trip_lookup(
                df_fus_cohort,
                trip_idx
            )

            if table is None or not lookup:
                print(
                    f"  WARNING: No supported distribution for "
                    f"trip {trip_idx}. Cohort cannot be generated."
                )
                supported = False
                break

            trip_tables[trip_idx] = table
            trip_lookups[trip_idx] = lookup

        if not supported:
            del df_fus_cohort
            gc.collect()
            continue

        # ----------------------------------------------------
        # Generate in bounded batches until target is reached.
        # No 18-million-person seed table is created.
        # ----------------------------------------------------
        completed_count = 0
        round_idx = 0

        while completed_count < target and round_idx < MAX_GENERATION_ROUNDS:
            round_idx += 1

            remaining = target - completed_count

            # Generate modest oversupply to compensate for unsupported
            # conditional transitions, but cap RAM with chunk size.
            batch_size = min(
                GENERATION_CHUNK_SIZE,
                max(
                    1,
                    int(np.ceil(remaining * 1.20))
                )
            )

            t0 = time.perf_counter()
            batch = generate_cohort_batch(
                df_fus_cohort=df_fus_cohort,
                tmax=tmax,
                batch_size=batch_size,
                next_start_lookup=next_start_lookup,
                trip_tables=trip_tables,
                trip_lookups=trip_lookups,
                rng=rng
            )
            generation_seconds += time.perf_counter() - t0

            generated = len(batch)

            print(
                f"  Round {round_idx:02d}: "
                f"seeded={batch_size:,}, "
                f"valid={generated:,}, "
                f"remaining={remaining:,}"
            )

            if generated == 0:
                print(
                    "  WARNING: Zero valid schedules generated. "
                    "Stopping this cohort."
                )
                break

            # We only retain the number needed for this cohort.
            keep = min(generated, remaining)

            if generated > keep:
                select_idx = rng.choice(
                    generated,
                    size=keep,
                    replace=False
                )
                batch = batch.iloc[select_idx].reset_index(drop=True)

            # Assign IDs now so row order and globally unique IDs remain stable.
            # Keep the compact encoded schedules; labels and coordinates are
            # restored only once after the full population has been assembled.
            n_batch = len(batch)
            batch.insert(0, "ID", np.arange(global_id, global_id + n_batch, dtype=np.int64))
            global_id += n_batch
            completed_count += n_batch
            total_generated += n_batch
            population_parts.append(batch)

            print(f"  → Retained {n_batch:,} schedules (cohort={completed_count:,}/{target:,}, cumulative={total_generated:,}/{int(N):,})")

        if completed_count == 0:
            print("  WARNING: No completed schedules for cohort.")
            del trip_tables, trip_lookups, df_fus_cohort
            gc.collect()
            continue

        if completed_count < target:
            print(
                f"  WARNING: target={target:,}, generated="
                f"{completed_count:,}. Shortfall={target-completed_count:,}"
            )

        del trip_tables, trip_lookups, df_fus_cohort

        # Do not force collection here because population_parts intentionally
        # retains all completed batches until the final concatenation.

    print("\nAssembling the final population DataFrame...")
    if population_parts:
        df_population = pd.concat(population_parts, ignore_index=True, sort=False, copy=False)
        del population_parts
        restoration_start = time.perf_counter()
        df_population = restore_coordinates_and_labels(df_population, zone_categories, coord_x, coord_y)
        restoration_seconds = time.perf_counter() - restoration_start
    else:
        df_population = pd.DataFrame(columns=schedule_columns_with_coordinates())
        restoration_seconds = 0.0

    print("\n" + "=" * 100)
    print("GENERATION COMPLETE")
    print("=" * 100)
    print(f"Target population : {int(N):,}")
    print(f"Generated population: {len(df_population):,}")
    print(f"Generation time   : {generation_seconds:.2f} seconds")
    print(f"Restoration time  : {restoration_seconds:.2f} seconds")

    if len(df_population) != int(N):
        print(
            f"WARNING: final population differs from target by "
            f"{int(N)-len(df_population):,}."
        )

    return df_population


# ============================================================
# Main
# ============================================================

if __name__ == "__main__":

    start_time = time.time()

    # --------------------------------------------------------
    # Parameters
    # --------------------------------------------------------


    rng = np.random.default_rng(RANDOM_SEED)

    print("=" * 100)
    print("RAM-EFFICIENT TOUR GENERATION")
    print("=" * 100)

    # --------------------------------------------------------
    # 1. Load HTS
    # --------------------------------------------------------
    print("\n[1/7] Loading HTS...")

    hts_cols = [
        "ACTIVITY_STARTTIME",
        "TRIP_PURPOSE",
        "ACTIVITY_DURATION"
    ]

    df_hts = pd.read_csv(
        os.path.join(
            path_data,
            f"data_{reg}_hts_trip.csv"
        ),
        usecols=hts_cols
    )

    df_hts = reduce_mem_usage(df_hts)

    print(
        f"HTS rows: {len(df_hts):,}, "
        f"memory={df_hts.memory_usage(deep=True).sum()/1024**2:.1f} MB"
    )

    # --------------------------------------------------------
    # 2. Load fused trip distribution
    # --------------------------------------------------------
    print("\n[2/7] Loading fused trip distribution...")

    fz = '05'
    fus_file = os.path.join(
        path_result,
        f"distort_pcm_case_{reg}_sim_trip_M_{KK}_nb_{nb}_seed_{RANDOM_SEED}_severity_{fz}_combined.csv"
    )

    required_fus_cols = [
        "AGE",
        "GENDER",
        "INCOME",
        "TRIP_MAX",
        "TRIP_CNT",
        "TRIP_PURPOSE",
        "TRAVEL_MODE",
        "ORIGIN_SUBZONE",
        "DESTINATION_SUBZONE",
        "TRIP_STARTTIME",
        "TRIP_ENDTIME",
        "Prob_XYZ_fus"
    ]

    # Only load columns actually required by schedule generation.
    # Coordinates are NOT loaded from df_fus.
    df_fus = pd.read_csv(
        fus_file,
        usecols=required_fus_cols
    )


    missing_cols = [
        c for c in required_fus_cols
        if c not in df_fus.columns
    ]

    if missing_cols:
        raise KeyError(
            f"Missing required fused-data columns: {missing_cols}"
        )

    df_fus = reduce_mem_usage(
        df_fus,
        use_cats=False
    )

    print(
        f"Fused rows: {len(df_fus):,}, "
        f"memory={df_fus.memory_usage(deep=True).sum()/1024**2:.1f} MB"
    )

    print(
        "Income categories:",
        sorted(
            df_fus["INCOME"]
            .dropna()
            .unique()
            .tolist()
        )
    )

    # --------------------------------------------------------
    # 3. Load PCM only for final coordinate restoration
    # --------------------------------------------------------
    print("\n[3/7] Loading PCM coordinate lookup...")

    coord_cols = [
        "ORIGIN_SUBZONE",
        "DESTINATION_SUBZONE",
        "ORIGIN_SUBZONE_X",
        "ORIGIN_SUBZONE_Y",
        "DESTINATION_SUBZONE_X",
        "DESTINATION_SUBZONE_Y"
    ]

    df_pcm = pd.read_csv(
        os.path.join(
            path_data,
            f"data_{reg}_pcm_trip.csv"
        ),
        usecols=coord_cols
    )

    # Coordinate table contains huge repeated OD rows.
    # Remove duplicate coordinate records before encoding.
    df_pcm = df_pcm.drop_duplicates(
        subset=[
            "ORIGIN_SUBZONE",
            "DESTINATION_SUBZONE",
            "ORIGIN_SUBZONE_X",
            "ORIGIN_SUBZONE_Y",
            "DESTINATION_SUBZONE_X",
            "DESTINATION_SUBZONE_Y"
        ]
    ).reset_index(drop=True)

    # --------------------------------------------------------
    # 4. Encode subzones
    # --------------------------------------------------------
    print("\n[4/7] Encoding subzones as int32...")

    df_fus, zone_categories, coord_x, coord_y = build_subzone_encoding(
        df_fus,
        df_pcm
    )

    del df_pcm
    gc.collect()

    print(
        f"Fused memory after subzone encoding: "
        f"{df_fus.memory_usage(deep=True).sum()/1024**2:.1f} MB"
    )

    # --------------------------------------------------------
    # 5. Build conditional next-start distribution
    # --------------------------------------------------------
    print("\n[5/7] Building next-start-time distribution...")

    df_next_start_times = next_start_times(
        df_hts
    )

    next_start_lookup = build_next_start_lookup(
        df_next_start_times
    )

    print(
        f"Next-start conditional groups: "
        f"{len(next_start_lookup):,}"
    )

    del df_hts, df_next_start_times
    gc.collect()

    # --------------------------------------------------------
    # 6. Derive exact cohort targets BEFORE generation
    # --------------------------------------------------------
    print("\n[6/7] Deriving AGE × GENDER × INCOME × TRIP_MAX targets...")

    cohort_targets = derive_cohort_targets(
        df_fus,
        int(NN)
    )

    print(
        f"Number of demographic/TRIP_MAX cohorts: "
        f"{len(cohort_targets):,}"
    )

    print(
        f"Allocated population: "
        f"{cohort_targets['Target'].sum():,}"
    )

    print("\nLargest cohort targets:")
    print(
        cohort_targets.sort_values(
            "Target",
            ascending=False
        ).head(20).to_string(index=False)
    )

    # --------------------------------------------------------
    # 7. Generate the complete DataFrame, then save exactly once
    # --------------------------------------------------------
    print("\n[7/7] Generating tours...")

    output_file = os.path.join(
        path_result,
        #f"case_{reg}_sim_tour_M_{KK}_nb_{nb}_seed_{RANDOM_SEED}.csv"
        f"distort_pcm_case_{reg}_sim_tour_M_{KK}_nb_{nb}_seed_{RANDOM_SEED}_severity_{fz}_combined.csv"

    )

    df_population = generate_population(
        df_fus=df_fus,
        cohort_targets=cohort_targets,
        next_start_lookup=next_start_lookup,
        zone_categories=zone_categories,
        coord_x=coord_x,
        coord_y=coord_y,
        N=int(NN),
        rng=rng
    )

    total_generated = len(df_population)

    # --------------------------------------------------------
    # Cleanup
    # --------------------------------------------------------
    del (
        df_fus,
        cohort_targets,
        next_start_lookup,
        zone_categories,
        coord_x,
        coord_y
    )

    gc.collect()

    # Save once only after the complete output DataFrame exists.
    output_file = os.path.abspath(os.path.expanduser(output_file))
    output_dir = os.path.dirname(output_file)
    os.makedirs(output_dir, exist_ok=True)
    if not os.access(output_dir, os.W_OK):
        raise PermissionError(f"Output directory is not writable: {output_dir}")

    print(f"\nSaving {total_generated:,} schedules to CSV in one operation...")
    save_start = time.perf_counter()
    df_population.to_csv(output_file, index=False)
    save_seconds = time.perf_counter() - save_start

    if not os.path.isfile(output_file) or os.path.getsize(output_file) == 0:
        raise OSError(f"Output validation failed: {output_file}")

    elapsed = time.time() - start_time

    print("\n" + "=" * 100)
    print("DONE")
    print("=" * 100)
    print(f"Output             : {output_file}")
    print(f"Generated schedules: {total_generated:,}")
    print(f"Target schedules   : {int(NN):,}")
    print(f"CSV writing time   : {save_seconds:.2f} seconds")
    print(f"Time taken         : {elapsed:.2f} seconds")
    print(f"Time taken         : {elapsed/60:.2f} minutes")

RAM-EFFICIENT TOUR GENERATION

[1/7] Loading HTS...
HTS rows: 39,122, memory=0.2 MB

[2/7] Loading fused trip distribution...
Fused rows: 120,737,980, memory=2418.0 MB
Income categories: [0, 1, 2]

[3/7] Loading PCM coordinate lookup...

[4/7] Encoding subzones as int32...
Number of unique subzones: 421
Fused memory after subzone encoding: 2418.0 MB

[5/7] Building next-start-time distribution...
Next-start conditional groups: 110

[6/7] Deriving AGE × GENDER × INCOME × TRIP_MAX targets...
Number of demographic/TRIP_MAX cohorts: 152
Allocated population: 9,000,000

Largest cohort targets:
 AGE  GENDER  INCOME  TRIP_MAX      Pop  Target
   2       1       1         2 0.085615  770534
   2       0       1         2 0.081982  737842
   1       1       1         2 0.057329  515962
   1       0       1         2 0.051232  461084
   2       1       1         3 0.033002  297020
   3       0       1         2 0.030544  274893
   1       1       0         2 0.030465  274183
   1       1       1